# 04. Retrieval 설계: 필요한 데이터를 잘 가져오는 법

**검색 단계에서 가져온 근거(Evidence)가 틀리면, 뒤따르는 모델이 아무리 뛰어나도 최종 답변은 틀립니다.** 이 노트북에서는 다섯 개의 짧은 문장으로 검색 점수를 직접 손으로 계산해 코드와 맞춰 보고, 어려운 질문 모음으로 검색 방식별 성능을 **숫자로 측정해 비교**합니다. 이어서 교차 인코더 재순위(Reranking), Decision API 관문(질문 분해와 출처 보강), 글자층이 없는 PDF 쪽 판독까지 차례대로 실험하고 측정합니다.

## 학습 목표

- 토큰화, BM25, 코사인 유사도, RRF(Reciprocal Rank Fusion)를 직접 손으로 계산해 코드 결과와 대조하고, 청킹(Chunking) 규칙과 고객·기준일 범위 필터가 검색 결과를 어떻게 바꾸는지 설명할 수 있습니다.
- 어려운 질문 모음에서 검색 방식별 재현율(recall)을 측정해 '누락(못 찾음)'과 '오인(범위 밖 문서를 가져옴)'을 구분하고, 표본 크기에 따른 해석의 한계를 짚어낼 수 있습니다.
- 초기 후보 깊이(pool)가 재현율(recall)의 상한선이 되는 이유와, 공개 벤치마크 1위 구성을 우리 문서에 그대로 옮기면 안 되는 이유를 측정 수치로 설명할 수 있습니다.
- 질문 분해를 Decision API 관문 뒤에서 기본으로 켜 둔 설계 이유를 이해하고, 질문 분해와 출처 보강이 실제 측정에서 어떤 효과를 보였는지 구분해 설명할 수 있습니다.

## 수업 시간표

| 절 | 내용 | 분 |
|---|---|---:|
| 1 | 왜 여전히 RAG인가, 근거 단위 | 8 |
| 2 | 다섯 문장의 토큰 | 2 |
| 3 | BM25의 세 요소 | 3 |
| 4 | Dense와 코사인 유사도 | 2 |
| 5 | RRF: 점수가 아니라 순위 | 3 |
| 6 | 청킹 규칙, 메타데이터는 자격 | 2 |
| 7 | 범위 필터 | 3 |
| 8 | 측정: recall, 누락과 오인 | 2 |
| 9 | 어려운 질문으로 측정 | 4 |
| 10 | 넓은 후보와 원 질문 재순위 | 7 |
| 11 | Jev 관문: 분해와 출처 보강 | 6 |
| 12 | AutoRAG와 다른 점 | 3 |
| 13 | PDF 쪽 판독 | 5 |
| - | 직접 해보기 | 5 |
| - | 확인 질문 | 3 |
| - | 정리 | 2 |

## 호출 규모

이 노트북에서 유료 Claude 모델 호출은 네 곳(1절의 문서 전체를 한 프롬프트에 넣은 질문, 관문이 열렸을 때의 질문 분해, PDF 쪽 판독, 직접 해보기의 내 질문 분해)에서 각각 최대 1회씩 일어납니다. 기본 경로의 모델 호출 합계는 **최대 4회로 예상**하며, 질문 분해는 관문이 닫히면 일어나지 않으므로 실제로는 이보다 줄어들 수 있습니다. Decision API(Jev) 호출은 11절의 관문 셀에서 3회와 1회, 직접 해보기에서 1회로 합계 **5회로 예상**하며 **질문 문장이 선택한 Decision API 공급자에게 전송됩니다.** 위 횟수는 모두 추정입니다. 유료 호출 직전에는 항상 호출 규모를 한 줄로 고지합니다.

**2026-11-02 실행 기록:** 저자가 이 노트북을 한 번, 한 대의 컴퓨터에서 끝까지 실행했을 때 Claude 호출은 3회(1절의 문서 전체 프롬프트 질문, 11절 관문이 열려 실행된 질문 분해, 13절 PDF 쪽 판독), Decision API(Jev) 호출은 5회(11절 관문 셀 3회와 1회, 직접 해보기 1회), 전체 소요 시간은 약 112초였습니다. 직접 해보기의 내 질문 관문은 닫혀 그 분해 호출은 일어나지 않았습니다. 한 번의 실행 기록이므로 관문의 `P` 값이나 모델 응답에 따라 호출 횟수와 시간은 달라질 수 있고, 로컬 모델을 처음 내려받는 시간은 실행 환경마다 다릅니다.

Dense 임베딩(`intfloat/multilingual-e5-small`)과 교차 인코더 재순위(`BAAI/bge-reranker-v2-m3`)는 로컬 모델로 동작하므로 API 호출 비용이 들지 않지만, **처음 실행할 때 모델 파일(각각 약 470MB, 2.1GB)을 내려받느라 시간이 걸릴 수 있습니다.**

각 코드 셀 첫 줄에 적힌 표지는 다음과 같습니다.
- `# [따라하기]`: 셀을 그대로 실행하고 결과를 관찰합니다.
- `# [바꿔보기]`: **결과를 먼저 예측해 적은 뒤** 값을 바꾸어 실행합니다.
- `# [깨뜨리기]`: 일부러 실패하거나 함정에 빠지는 조건을 만들어 원인을 확인합니다.
- `# [내 것 만들기]`: 예시 값을 내 질문과 문장으로 바꾸어 실험합니다.

먼저 아래 준비 셀을 실행해 필요한 모듈을 불러오고 API 키 상태를 확인합니다.

In [ ]:
# 이 노트북 전체에서 쓰는 모듈과 함수를 한 번에 불러옵니다.
from datetime import date
from math import log
from time import perf_counter

import numpy as np
import pymupdf
from kiwipiepy import Kiwi

from rag_lab.decisions import make_decider
from rag_lab.corpus import Chunk, Scope, count_cut_tables, fixed_chunks, load_cases, load_corpus, section_chunks
from rag_lab.evaluation import evaluate_retrieval, load_gold, recall_by_type
from rag_lab.evidence import RetrievalContext
from rag_lab.gates import needs_decomposition
from rag_lab.pages import needs_reading, read_page
from rag_lab.pipeline import decompose_question, retrieve
from rag_lab.retrieval import BM25Index, Retriever, rrf, tokenize
from rag_lab.settings import ROOT, announce, key_status, make_embeddings, make_model, make_reranker, require_key

require_key()  # API 키가 없으면 여기서 멈추고 .env 파일에 채우는 방법을 안내합니다
print(key_status())

## 1. 왜 여전히 RAG인가, 근거 단위 (8분)

이 절은 "모델이 이렇게 똑똑한데 검색을 왜 따로 설계해야 하는가"라는 질문에 **이 수업의 문서 말뭉치**로 답합니다. 일반론이 아니라 아래 코드 셀에서 직접 계산하고 출력해 보며 따라갑니다.

### 첫째, 모델은 우리 회사의 내부 문서를 알지 못합니다

아무리 뛰어난 모델이라도 **우리 회사의 내부 계약서와 사내 규정은 알지 못합니다.** 모델의 지식은 사전 학습 데이터에서 오는데, 사내 계약서와 규정은 그 데이터에 들어 있지 않습니다. `01-Agentic-AI-Basics.ipynb`의 단일 모델 호출 셀에서 이번 주 사내 식당 메뉴를 물었던 것도 같은 문제였습니다. 모델이 모르는 정보는 **모델 바깥에서 찾아 모델에게 건네야** 합니다. 그 일을 하는 것이 검색(Retrieval)입니다.

### 둘째, 그렇다면 문서를 프롬프트에 통째로 넣으면 안 될까요

가장 단순한 대안은 사내 문서 **전체**를 프롬프트에 붙여 넣고 질문하는 것입니다. 이 방식이 왜 문제가 되는지 두 가지로 나누어 봅니다.

1. **크기와 비용**: 프롬프트가 커질수록 호출 비용과 응답 시간이 늘고, 문서가 늘어나면 모델이 한 번에 읽을 수 있는 길이의 한도에 부딪힙니다. 우리 수업 말뭉치는 일부러 작게 만들었지만, 실제 회사의 문서는 이보다 훨씬 많습니다.
2. **자격**: 문서 전체에는 질문과 상관없는 문서만 있는 것이 아닙니다. 이미 만료된 옛 버전, 승인 전 초안, 다른 고객의 계약서가 함께 들어 있습니다. 이 문제는 크기를 줄여서는 풀리지 않습니다.

먼저 크기부터 직접 계산해 봅니다. 아래 셀은 모델을 부르지 않는 무료 셀입니다. 문서 원문 전체의 UTF-8 바이트 크기(`whole_corpus_bytes`)와, 같은 질문에 대해 고객·기준일 범위 안에서 BM25로 고른 상위 3개 조각의 바이트 크기를 나란히 출력합니다. 이 셀은 `# [깨뜨리기]` 표지가 붙어 있습니다. 통째로 넣는 방식의 약점을 일부러 드러내 보는 셀이기 때문입니다.

In [ ]:
# [깨뜨리기] 문서를 프롬프트에 통째로 넣을 때와 질문에 맞는 조각만 넣을 때의 바이트 크기를 비교합니다 (무료, 모델 호출 0회)

# [1단계] 수업 문서 말뭉치를 모두 불러옵니다. 문서 하나가 Document 하나입니다
documents = load_corpus()

# [2단계] 통째로 넣는 경우: 모든 문서 원문을 UTF-8 바이트로 바꾸어 길이를 더합니다
whole_corpus_bytes = 0
for doc in documents.values():
    whole_corpus_bytes += len(doc.text.encode("utf-8"))

# [3단계] 질문에 맞게 넣는 경우: 고객과 기준일 범위 안에서 BM25 상위 3개 조각만 고릅니다
QUESTION = "알파 서비스 크레딧 신청 기한은 언제인가요?"
scope = Scope("알파", date(2026, 9, 8))
retriever_bm25 = Retriever(documents)  # 기본 검색 방식은 bm25이므로 임베딩 모델을 적재하지 않습니다
top_chunk_ids = retriever_bm25.rank(QUESTION, scope, limit=3)

top_chunks_bytes = 0
for chunk_id in top_chunk_ids:
    chunk = retriever_bm25.chunks[chunk_id]
    top_chunks_bytes += len(chunk.text.encode("utf-8"))

# [4단계] 두 크기를 같은 단위(바이트)로 나란히 출력합니다
print(f"문서 {len(documents)}개 전체를 통째로 넣을 때: {whole_corpus_bytes} 바이트 (whole_corpus_bytes)")
print(f"범위 안 상위 {len(top_chunk_ids)}개 조각만 넣을 때: {top_chunks_bytes} 바이트  {top_chunk_ids}")
print(f"통째로 넣는 쪽이 조각만 넣는 쪽의 {whole_corpus_bytes / top_chunks_bytes:.1f}배입니다")

출력의 두 숫자를 비교해 보세요. 프롬프트에 실제로 필요한 분량이 문서 전체 대비 어느 정도인지 읽을 수 있습니다. 다만 이 말뭉치는 수업용으로 작게 만든 것이므로, 숫자의 **절대 크기**보다 **문서 수가 늘어날 때 두 숫자가 각각 어떻게 변하는지**를 생각해 보는 것이 중요합니다. 통째로 넣는 쪽은 문서가 늘면 그대로 커지지만, 질문에 맞는 조각만 고르는 쪽은 `limit`로 정한 조각 수만큼만 커집니다.

### 셋째, 크기보다 더 큰 문제는 자격입니다

바이트가 작다고 해서 통째로 넣어도 되는 것은 아닙니다. 같은 질문 `알파 서비스 크레딧 신청 기한은 언제인가요?`를 한 고객(`알파`)이 기준일 `2026-09-08`에 묻는다고 합시다. 문서 전체를 넣은 프롬프트에는 알파 서비스 계약서의 여러 버전(`sla-a-v1`, `sla-a-v2`), 추가 약정(`sla-a-amendment`), 승인 전 초안(`sla-a-draft`), 그리고 다른 고객의 계약서(`sla-b`)가 모두 들어갑니다.

문서에는 `Document`의 필드로 승인 상태(`status`), 고객(`entity`), 유효 기간(`valid_from`, `valid_until`)이 붙어 있습니다(`rag_lab/corpus.py:27`). 그리고 `Document.applies(scope)`(`rag_lab/corpus.py:41`)가 이 필드들로 "이 고객이 이 기준일에 이 문서를 근거로 써도 되는가"를 판정합니다. 아래 셀은 앞의 다섯 문서를 하나씩 꺼내 상태, 고객, 유효 기간과 `applies`의 판정을 나란히 출력합니다. 출력을 읽으며 **통째로 넣은 프롬프트 안에서 어느 문서가 자격이 있고 어느 문서가 없는지** 직접 구분해 보세요.

In [ ]:
# [깨뜨리기] 통째로 넣은 프롬프트에 섞여 들어갈 문서들의 상태, 고객, 유효 기간과 질문 범위의 자격을 확인합니다 (무료, 모델 호출 0회)

# 알파 서비스 계약서의 버전들, 추가 약정, 초안, 다른 고객(베타)의 계약서를 질문 범위(scope)로 하나씩 판정합니다
MIXED_DOC_IDS = ["sla-a-v1", "sla-a-v2", "sla-a-amendment", "sla-a-draft", "sla-b"]

print(f"질문 범위: 고객={scope.entity}, 기준일={scope.as_of}")
for doc_id in MIXED_DOC_IDS:
    doc = documents[doc_id]
    # valid_until이 없으면 끝나는 날이 정해지지 않은 문서입니다. 있다면 그날은 유효 기간에서 제외됩니다
    valid_until = doc.valid_until if doc.valid_until else "(끝나는 날 없음)"
    qualified = doc.applies(scope)
    print(
        f"{doc_id:16s} | 상태={doc.status:8s} | 고객={doc.entity} | "
        f"유효 {doc.valid_from} ~ {valid_until} | 버전={doc.version} | 자격(applies)={qualified}"
    )

출력의 마지막 열이 `False`인 문서는 이 질문의 근거로 쓸 자격이 없습니다. 그런데 통째로 넣은 프롬프트에서는 자격이 있는 문서와 없는 문서가 **같은 자리에 같은 모양으로** 놓입니다. 모델이 이 구분을 스스로 해내리라고 기대하는 것은 설계가 아니라 기대일 뿐입니다. 자격은 모델이 읽기 전에 **코드가** 걸러야 합니다. 이것이 7절에서 배울 범위 필터의 출발점입니다.

### 넷째, 근거 단위, 최신성, 접근 통제

통째로 넣는 방식이 놓치는 것은 크기와 자격만이 아닙니다. 운영 환경에서 RAG를 쓰는 이유를 세 가지 요구로 정리하면 다음과 같습니다.

- **인용할 수 있는 근거 단위**: 답변이 "계약서에 그렇게 적혀 있다"라고 말하려면 어느 문서의 어느 줄인지 가리킬 수 있어야 합니다. 이 저장소의 조각(`Chunk`)은 원문 안의 문자 위치(`start`, `end`)와 행 번호(`start_line`, `end_line`)를 함께 가지고 있어(`rag_lab/corpus.py:51`) 답변에서 줄 단위로 인용할 수 있습니다. 문서 전체를 이어 붙인 한 덩어리 프롬프트에는 이런 단위가 없습니다. 근거(Evidence) 하나를 어떤 크기로 자를지는 6절에서 다룹니다. 잘라 낸 조각 하나만 읽어도 앞뒤 맥락과 뜻이 온전히 통해야 올바른 근거로 인용할 수 있기 때문입니다.
- **최신성**: 계약서와 규정은 개정됩니다. 만료된 버전을 근거로 답하면 틀린 답이 되고, 모델은 틀렸다는 사실을 알지 못합니다. 기준일(`as_of`)을 검색 단계에서 문서의 유효 기간과 맞대어 걸러야 합니다. 문서가 바뀌면 다시 적재(ingest)하고, 검색은 항상 현재 말뭉치를 대상으로 해야 합니다.
- **접근 통제**: 다른 고객의 계약서는 이 고객에게 보여 주면 안 되는 정보일 수 있습니다. 한 번 프롬프트에 들어간 내용은 답변에 드러날 수 있으므로, 접근 권한이 없는 문서는 **프롬프트에 들어가기 전에** 빠져야 합니다. 모델에게 "다른 고객 문서는 무시해"라고 부탁하는 것은 통제가 아닙니다.

정리하면 RAG의 일은 단순히 "문서를 찾아 주는 것"이 아닙니다. **질문에 필요한 조각만, 현재 고객과 기준일에 자격이 있는 문서에서만, 나중에 출처를 인용할 수 있는 온전한 모양으로** 가져오는 것입니다. 이 일은 모델 바깥에서 우리가 직접 설계해야 합니다. 그 출발점은 **'근거(Evidence) 하나를 어떤 단위로 자를 것인가'**를 정하는 일입니다.

### 직접 확인해 보기: 문서를 전부 넣고 물으면 어떻게 답할까요

아래 `# [깨뜨리기]` 셀은 유료 셀입니다. 문서 전체를 한 프롬프트에 넣고, 위와 같은 질문을 고객과 기준일(`2026-09-08`)과 함께 모델에게 한 번 묻습니다. 모델 호출은 1회이며 실행 직전에 `announce`가 규모를 한 줄로 고지합니다.

In [ ]:
# [깨뜨리기] 문서 전체를 한 프롬프트에 넣고 같은 질문을 기준일과 함께 묻습니다 (유료, 모델 호출 1회)

# [1단계] 모든 문서 원문을 문서 ID와 제목 머리글을 붙여 하나의 긴 글로 이어 붙입니다
document_blocks = []
for doc in documents.values():
    document_blocks.append(f"## 문서 {doc.doc_id}: {doc.title}\n{doc.text}")
whole_corpus_text = "\n\n".join(document_blocks)

# [2단계] 문서 전체 뒤에 질문, 고객, 기준일을 덧붙여 프롬프트 하나를 만듭니다
whole_prompt = (
    "아래는 회사 문서 전체입니다. 문서만 근거로 질문에 답하세요.\n\n"
    f"{whole_corpus_text}\n\n"
    f"질문: {QUESTION}\n"
    f"고객: {scope.entity}\n"
    f"기준일: {scope.as_of.isoformat()}"
)
print(f"프롬프트 크기: {len(whole_prompt.encode('utf-8'))} 바이트")

# [3단계] 모델 호출 1회를 고지한 뒤 질문합니다
announce("문서 전체를 한 프롬프트에 넣고 질문합니다", model_calls=1)
reply = make_model().invoke(whole_prompt)
print("[모델 답변]")
print(reply.text)

모델의 답변을 읽고 다음 두 질문에 직접 답해 보세요. 결과를 미리 단정하지 말고 **출력에 나온 내용만** 근거로 삼습니다.

1. 답변은 앞에서 확인한 여러 버전의 문서 가운데 **어느 버전**을 근거로 썼습니까? 그 버전은 기준일에 유효한 문서입니까?
2. 답변에서 **문서 ID와 줄 번호**를 가리켜 "이 줄이 근거"라고 인용할 수 있습니까? 모델이 문서 ID를 적었다면, 그 줄이 실제 원문에 있는지 지금 코드로 확인할 방법이 있습니까?

두 질문의 답이 어떻게 나왔든, 문서 전체를 넣는 방식에서는 **어떤 문서가 답에 쓰였는지를 코드가 통제하지 못한다**는 점은 같습니다. 이 노트북의 나머지 절은 그 통제를 코드로 가져오는 방법(토큰화, BM25, Dense, RRF, 청킹, 범위 필터, 측정)을 차례로 배웁니다.

## 2. 다섯 문장의 토큰 (2분)

키워드 검색기는 문장 전체가 아니라 쪼개진 **토큰(Token)** 단위로 낱말이 겹치는지 비교합니다. 한국어에서는 `식당은`, `식당을`, `식당에서`처럼 명사 뒤에 조사가 붙기 때문에 단순히 띄어쓰기(공백)로만 나누면 서로 다른 단어로 취급됩니다. 반면 한국어 형태소 분석기인 `Kiwi`를 쓰면 명사와 조사를 깔끔하게 분리해 줍니다. 아래 셀은 모델을 부르지 않는 무료 셀입니다.

In [ ]:
# [따라하기] 공백 분리와 형태소 분석기(Kiwi) 토큰화를 비교합니다 (무료, 모델 호출 0회)

# [1단계] 업무 문장 다섯 개를 준비합니다. 앞쪽 절들에서 같은 문장을 계속 씁니다
SENTENCES = [
    "사내 식당은 오전 11시 30분에 문을 엽니다.",
    "도서관 책은 2주 동안 빌릴 수 있습니다.",
    "회의실은 하루 전까지 예약해야 합니다.",
    "식당 메뉴는 매주 월요일에 공지합니다.",
    "통근 버스는 오후 6시에 출발합니다.",
]

# [2단계] 형태소 분석기를 만들고 첫 문장 하나를 두 가지 방법으로 나눕니다
kiwi = Kiwi()
first_sentence = SENTENCES[0]
whitespace_tokens = first_sentence.split()
kiwi_tokens = []
for token in kiwi.tokenize(first_sentence):
    kiwi_tokens.append((token.form, token.tag))  # (형태소, 품사 태그)

# [3단계] 두 결과를 나란히 출력합니다
print("공백 분리:", whitespace_tokens)
print("Kiwi 분석:", kiwi_tokens)

위 출력에서 볼 수 있듯, `Kiwi`는 `식당은`을 형태소 `식당`(`NNG`, 일반명사)과 `은`(`JX`, 보조사)으로 나눕니다. 우리 검색기가 쓰는 `tokenize` 함수(`rag_lab/retrieval.py:25`)는 형태소 분석 결과에서 조사를 버리고 의미를 담은 품사(명사 `N*`, 동사·형용사 `V*`, 영문 `SL`, 숫자 `SN`)만 남깁니다.

이제 아래 `[바꿔보기]` 셀에서 문장 속 표현을 `식당은`, `식당을`, `식당에서`로 바꾸어 보겠습니다. **먼저 예측해 보세요:** 세 문장의 공백 분리 결과는 서로 다른데, `tokenize` 함수가 뽑아낸 첫 번째 토큰은 같을까요, 다를까요?

In [ ]:
# [바꿔보기] 바꿀 값: VARIANTS 리스트의 문장들. 결과를 먼저 예측하고 실행하세요. (무료)
VARIANTS = ["식당은 열려 있나요", "식당을 찾습니다", "식당에서 먹습니다"]

# 문장마다 공백 분리 결과와 우리 검색기의 tokenize 결과를 나란히 출력합니다
for text in VARIANTS:
    whitespace_tokens = text.split()
    morpheme_tokens = tokenize(text)
    print(f"{text!r:18} -> 공백 분리={whitespace_tokens} | tokenize={morpheme_tokens}")

조사가 달라져도 `tokenize` 결과에서는 모두 `'식당'`이라는 동일한 토큰으로 뽑히는 것을 확인했습니다.

## 3. BM25의 세 요소 (3분)

키워드 검색의 표준인 **BM25** 점수는 다음 세 가지 요소로 결정됩니다(`rag_lab/retrieval.py:36`).
- **IDF (역문서 빈도)**: 여러 조각에 흔하게 나오는 단어일수록 점수가 낮고, 소수의 조각에만 드물게 나오는 단어일수록 점수가 큽니다. 이 저장소의 `BM25Index`는 `log(1 + (N - df + 0.5) / (df + 0.5))` 공식을 써서 IDF가 항상 0보다 크게 만듭니다(여기서 `N`은 전체 조각 수, `df`는 해당 단어가 등장한 조각 수입니다).
- **TF (단어 빈도)**: 한 조각 안에 같은 단어가 여러 번 나올수록 점수가 올라가지만, 일정 수준을 넘으면 증가 폭이 완만하게 포화됩니다(`k1 = 1.5`).
- **문서 길이 보정**: 전체 평균 길이보다 유난히 긴 조각은 우연히 단어가 겹칠 확률이 높으므로 점수를 깎아 보정합니다(`b = 0.75`).

아래 셀을 실행해 `'식당'`의 IDF 값을 손으로 직접 계산해 보고, `BM25Index`가 계산한 값과 일치하는지 확인해 보세요.

In [ ]:
# [따라하기] '식당'의 IDF를 손으로 계산해 BM25Index 색인 값과 맞추고 검색합니다 (무료)

# [1단계] 다섯 문장을 조각(Chunk) 다섯 개로 감싸고 BM25 색인을 만듭니다
chunks = []
for i, sentence in enumerate(SENTENCES):
    chunk = Chunk(f"s{i}", "notes", "", sentence, 0, len(sentence), 1, 1)
    chunks.append(chunk)
index = BM25Index(chunks)

# [2단계] IDF를 손으로 계산합니다. N은 전체 조각 수, df는 '식당'이 들어 있는 조각 수입니다
N = len(chunks)
df = 0
for chunk in chunks:
    if "식당" in tokenize(chunk.text):
        df += 1
by_hand = log(1 + (N - df + 0.5) / (df + 0.5))
index_idf = index.idf["식당"]
print(f"전체 조각 N={N}, '식당' 등장 조각 df={df} -> 손계산 IDF={by_hand:.4f}, 색인 IDF={index_idf:.4f}")
print()

# [3단계] 질의로 검색해 점수가 높은 순서로 세 개를 출력합니다
for chunk_id, score in index.search("식당 메뉴 언제 공지해?", limit=3):
    sentence = SENTENCES[int(chunk_id[1:])]  # chunk_id가 's3'이면 3번째 문장입니다
    print(f"{chunk_id} (점수 {score:.3f}): {sentence}")

우리가 공식대로 손으로 계산한 IDF 값과 `BM25Index` 내부의 IDF 값이 정확히 일치합니다. 검색 결과를 보면 질의 단어(`식당`, `메뉴`, `공지`)가 세 개나 겹친 `s3` 문장이 1위이고, `식당` 하나만 겹친 `s0` 문장이 2위로 내려갑니다.

이어서 BM25의 나머지 두 요소인 **단어 빈도(TF) 포화**와 **문서 길이 보정**도 짧은 코드로 직접 확인해 봅시다. 같은 단어(`식당`)를 1번, 2번, 5번 반복한 문장과, 쓸데없는 수식어로 길이만 길어진 문장의 점수가 어떻게 매겨지는지 비교합니다.


In [ ]:
# [따라하기] 무료, 모델 호출 0회 — BM25의 단어 빈도(TF) 포화와 문서 길이 보정 확인하기

# [1단계] '식당'을 1번, 2번, 5번 쓴 문장과 길이만 긴 문장, 무관한 문장을 준비합니다
tf_test_texts = [
    ("TF 1회 (짧은 문장)", "식당 안내"),
    ("TF 2회 (짧은 문장)", "식당 식당"),
    ("TF 5회 (반복 문장)", "식당 식당 식당 식당 식당"),
    ("TF 1회 (긴 문장)  ", "식당 안내입니다 오늘 날씨와 사내 게시판 일반 공지사항 안내문을 함께 적어 길이를 늘린 문장"),
    ("무관한 기준 문장  ", "도서관 책 대여 안내"),
]

# [2단계] 라벨을 조각 ID로 삼아 이 문장들만의 BM25 색인을 만듭니다
tf_chunks = []
for label, text in tf_test_texts:
    tf_chunks.append(Chunk(label, "notes", "", text, 0, len(text), 1, 1))
tf_index = BM25Index(tf_chunks)

# [3단계] '식당'으로 검색해 문장별 점수를 출력합니다
for label, score in tf_index.search("식당", limit=5):
    print(f"{label} -> BM25 점수: {score:.4f}")

**관찰할 포인트:**
- **TF 포화:** `식당`을 1번 쓴 문장보다 2번, 5번 쓴 문장의 점수가 올라가지만, 점수가 2배·5배로 정비례해서 폭주하지 않고 완만하게 수렴합니다(`k1 = 1.5`).
- **문서 길이 보정:** 똑같이 `식당`이 1번 들어 있어도, 핵심만 짧은 문장(`TF 1회 (짧은 문장)`)이 군더더기가 많은 긴 문장(`TF 1회 (긴 문장)`)보다 훨씬 높은 점수를 받습니다(`b = 0.75`).

**[바꿔보기]** 이제 원래 다섯 문장 색인(`index`)으로 돌아와 검색 질의를 `"대출 기간"`으로 바꾸어 보겠습니다. 다섯 문장 가운데 도서관 문장(`s1`)에는 `"빌릴 수 있습니다"`라고만 적혀 있고 `"대출"`이라는 단어는 없습니다. **먼저 예측해 보세요:** BM25가 도서관 문장을 찾아낼 수 있을까요?


In [ ]:
# [바꿔보기] 바꿀 값: QUERY. 예측을 먼저 적고 실행하세요. (무료)
QUERY = "대출 기간"

query_tokens = tokenize(QUERY)
bm25_hits = index.search(QUERY, limit=3)
print("질의 토큰:", query_tokens)
print("BM25 검색 결과:", bm25_hits)

검색 결과가 빈 리스트(`[]`)로 나왔습니다. 질의에서 뽑은 토큰(`['대출', '기간']`)이 다섯 문장 어디에도 글자 그대로 들어 있지 않기 때문입니다. 이처럼 **같은 뜻이라도 표현하는 낱말이 다르면(어휘 불일치)** BM25는 후보를 하나도 만들지 못합니다. 바로 이 한계를 보완하기 위해 의미 벡터를 비교하는 Dense 검색이 필요합니다.

잠깐, 왜 우리는 널리 쓰이는 `rank_bm25` 패키지의 `BM25Okapi`를 그대로 쓰지 않고 `BM25Index`를 직접 구현했을까요? `BM25Okapi`는 어떤 단어가 전체 조각의 절반을 넘는 곳에서 등장하면 IDF가 **음수(또는 0)**가 됩니다. 만약 점수가 0보다 큰 후보만 남기는 필터(`score > 0`)와 만나면 정답 후보가 어떻게 되는지 문장 한 개짜리 색인으로 확인해 봅시다.

In [ ]:
# [깨뜨리기] 문장 하나만 넣은 BM25Okapi와 우리 BM25Index를 비교합니다 (무료)
from rank_bm25 import BM25Okapi

# [1단계] 같은 문장 하나(s3)로 두 가지 색인을 만듭니다
one = SENTENCES[3]
okapi = BM25Okapi([tokenize(one)])
mine = BM25Index([Chunk("one", "notes", "", one, 0, len(one), 1, 1)])

# [2단계] 같은 질의를 두 색인에 넣어 점수를 얻습니다
query = "식당 메뉴"
okapi_scores = okapi.get_scores(tokenize(query))

# [3단계] BM25Okapi의 IDF와 점수, 그리고 score > 0 필터를 통과하는 후보를 확인합니다
okapi_idf = {}
for token, value in okapi.idf.items():
    okapi_idf[token] = round(value, 3)

positive_scores = []
for score in okapi_scores:
    if score > 0:
        positive_scores.append(score)

print("BM25Okapi IDF :", okapi_idf)
print("BM25Okapi 점수:", okapi_scores, "-> 0보다 큰 후보:", positive_scores)
print("BM25Index 점수:", mine.search(query))

위 출력에서 볼 수 있듯, `BM25Okapi`에서는 단어가 1개 중 1개 문장에 모두 들어 있으므로 IDF가 음수가 되고, 총점도 음수가 되어 `score > 0` 필터를 거치면 정답 후보가 **아무런 오류 메시지 없이 조용히 사라져 버립니다.** 반면 양수 IDF 공식을 쓴 `BM25Index`는 정상적으로 양수 점수를 부여해 후보를 지킵니다.

## 4. Dense와 코사인 유사도 (2분)

**Dense 검색**은 문장과 질의를 고정 길이의 실수 벡터(임베딩)로 바꾼 뒤, 두 벡터가 가리키는 방향이 얼마나 가까운지(**코사인 유사도**)를 계산해 **표현이 달라도 뜻이 비슷한 문장**을 찾습니다. 임베딩 모델은 답변을 생성하는 LLM과는 **별개로 선택하는 부품**이며, 이 수업에서는 다국어 모델인 `intfloat/multilingual-e5-small`(384차원)을 로컬 CPU에서 실행합니다.

In [ ]:
# [따라하기] 로컬 임베딩 모델로 코사인 유사도를 직접 계산합니다 (무료, 첫 실행 시 모델 다운로드로 시간이 걸립니다)

# [1단계] 임베딩 모델을 만들고 다섯 문장과 질의를 각각 벡터로 바꿉니다
embeddings = make_embeddings()
vectors = np.array(embeddings.embed_documents(SENTENCES))
query_vector = np.array(embeddings.embed_query("대출 기간"))

# [2단계] 벡터 길이(L2 norm)가 모두 1인지 확인합니다. 1이면 내적이 곧 코사인 유사도입니다
vector_lengths = np.linalg.norm(vectors, axis=1).round(3)
print("문장 벡터 행렬 모양:", vectors.shape, "| 각 벡터의 길이(L2 norm):", vector_lengths)

# [3단계] 문장 벡터 행렬과 질의 벡터를 내적하고 유사도가 높은 순서로 출력합니다
scores = vectors @ query_vector
for i in np.argsort(-scores):
    print(f"유사도 {scores[i]:.3f} | {SENTENCES[i]}")

모든 벡터의 길이(L2 norm)가 `1.0`으로 정규화되어 있으므로, 행렬 내적(`vectors @ query_vector`)을 계산한 값이 곧 코사인 유사도가 됩니다. 앞서 BM25가 한 건도 찾지 못했던 `"대출 기간"` 질의에서 Dense 검색은 도서관 문장(`"2주 동안 빌릴 수 있습니다"`)을 1위로 찾아냅니다. 하지만 모든 문장에 서로 비슷한 점수(위 출력에서는 0.767~0.821)가 매겨지고 1위와 2위의 점수 차이가 작을 때는 순위가 쉽게 흔들릴 수 있습니다.

아래 `[바꿔보기]` 셀에서는 질의를 `"점심 먹는 곳 여는 시간"`으로 바꾸어 보겠습니다. **먼저 예측해 보세요:** `"사내 식당은 오전 11시 30분에 문을 엽니다"` 문장이 1위로 올라올까요?

In [ ]:
# [바꿔보기] 바꿀 값: QUERY. 예측을 먼저 적고 실행하세요. (무료)
QUERY = "점심 먹는 곳 여는 시간"

# [1단계] 질의를 벡터로 바꾸어 다섯 문장과의 코사인 유사도를 구하고 높은 순서로 출력합니다
scores = vectors @ np.array(embeddings.embed_query(QUERY))
order = np.argsort(-scores)
for i in order:
    print(f"유사도 {scores[i]:.3f} | {SENTENCES[i]}")

# [2단계] 1위와 2위의 점수 차이를 보고, 같은 질의의 BM25 결과와 비교합니다
gap = float(scores[order[0]] - scores[order[1]])
print("1위와 2위의 점수 차이:", round(gap, 3))
print("참고 — 같은 질의의 BM25 결과:", index.search(QUERY, limit=3))

출력 결과를 보면 Dense 검색도 완벽하지 않다는 것을 알 수 있습니다. `"점심 먹는 곳"`이라는 표현이 `"식당 메뉴"` 문장과도 의미상 가깝다고 판단해, 정작 여는 시간을 알려 주는 영업시간 문장과 점수가 거의 비등하거나 순서가 뒤집히기도 합니다(반면 BM25는 `'열다'`의 활용형 어간 `'열'` 토큰 하나로 영업시간 문장 `s0`을 정확히 1위로 찾았습니다).

## 5. RRF: 점수가 아니라 순위 (3분)

그렇다면 키워드에 강한 BM25와 의미에 강한 Dense를 어떻게 합쳐야 할까요? BM25 점수는 상한이 없는 양수이고 코사인 유사도는 `-1 ~ 1`(E5에서는 보통 `0.7 ~ 0.9`) 사이에 몰려 있어 **점수의 단위와 분포가 전혀 다릅니다.**

그래서 **RRF(Reciprocal Rank Fusion)**는 원래 점수를 버리고 각 검색기가 매긴 **순위(Rank)**만 사용합니다(`rag_lab/retrieval.py:79`). 각 순위 목록에서 `가중치 / (상수 + 순위)`를 계산해 더하며, 완화 상수(`constant`)는 기본값으로 `60`을 씁니다.

In [ ]:
# [따라하기] RRF 점수를 손으로 계산하고 rrf() 함수 결과와 대조합니다 (무료)
QUERY = "점심 먹는 곳 여는 시간"

# [1단계] BM25 순위 목록과 Dense(코사인 유사도) 순위 목록을 각각 구합니다
bm25_rank = []
for chunk_id, _ in index.search(QUERY, limit=5):
    bm25_rank.append(chunk_id)

dense_scores = vectors @ np.array(embeddings.embed_query(QUERY))
dense_rank = []
for i in np.argsort(-dense_scores):
    dense_rank.append(chunks[i].chunk_id)

# [2단계] 각 목록에서의 순위(rank, 1부터 시작)로 1 / (60 + rank) 점수를 더해 RRF 합산 점수를 계산합니다
table = {}
for ranking in (bm25_rank, dense_rank):
    for rank, chunk_id in enumerate(ranking, 1):
        table[chunk_id] = table.get(chunk_id, 0.0) + 1.0 / (60 + rank)

def score_then_id(chunk_id):
    """정렬 기준: 합산 점수가 높은 조각이 먼저, 점수가 같으면 조각 ID 순서로 정렬합니다."""
    return (-table[chunk_id], chunk_id)

by_hand = sorted(table, key=score_then_id)

# [3단계] 각 조각의 순위와 합산 점수를 출력하고 rrf() 함수 결과와 일치하는지 확인합니다
print("BM25 순위 목록 :", bm25_rank)
print("Dense 순위 목록:", dense_rank)
for chunk_id in by_hand:
    if chunk_id in bm25_rank:
        b_pos = f"{bm25_rank.index(chunk_id) + 1}위"
    else:
        b_pos = "없음"
    d_pos = f"{dense_rank.index(chunk_id) + 1}위"
    print(f"  {chunk_id}: BM25 {b_pos:4s}, Dense {d_pos:3s} -> RRF 합산 점수 {table[chunk_id]:.5f}")
print("손계산 순서와 rrf() 결과가 일치하는가:", by_hand == rrf([bm25_rank, dense_rank]))

Dense에서는 2위로 밀렸던 `s0`(식당 여는 시간)가 BM25에서도 1위로 등장한 덕분에, 두 목록의 순위 점수를 합친 RRF에서는 당당히 단독 1위로 올라왔습니다.

이번에는 아래 `[바꿔보기]` 셀에서 완화 상수 `CONSTANT`를 `1`로 바꾸거나, 가중치 `WEIGHTS`를 `[0.0, 1.0]`(BM25 몫을 0으로 제거)으로 바꾸어 봅니다. **먼저 예측해 보세요:** 최종 순위가 어떻게 바뀔까요?

In [ ]:
# [바꿔보기] 바꿀 값: CONSTANT와 WEIGHTS. 예측을 먼저 적고 실행하세요. (무료)
CONSTANT = 1
WEIGHTS = [0.0, 1.0]  # [BM25 몫, Dense 몫]

default_order = rrf([bm25_rank, dense_rank])
constant_order = rrf([bm25_rank, dense_rank], constant=CONSTANT)
weights_order = rrf([bm25_rank, dense_rank], weights=WEIGHTS)

print("기본값 (constant=60, 동일 가중치):", default_order)
print("상수만 변경 (constant=1)         :", constant_order)
print("가중치만 변경 (weights=[0.0, 1.0]):", weights_order)

위 출력에서 볼 수 있듯 가중치를 `[0.0, 1.0]`으로 두면 BM25의 기여가 완전히 사라져 Dense 단독 순위와 똑같아집니다.

그렇다면 왜 이렇게 번거롭게 순위(RRF)를 쓰거나 정규화를 거쳐야 할까요? 아래 `[깨뜨리기]` 셀을 실행해 가중치 개수를 잘못 넣었을 때의 예외와, **BM25 원점수와 코사인 유사도 원점수를 정규화 없이 그냥 더했을 때** 어떤 일이 벌어지는지 확인해 보세요.

In [ ]:
# [깨뜨리기] 가중치 개수가 목록 개수와 다를 때의 오류와, 정규화 없이 원점수를 그냥 더했을 때의 문제를 확인합니다 (무료)

# [1단계] 순위 목록은 하나인데 가중치를 두 개 넣으면 ValueError가 납니다
try:
    rrf([["a"]], weights=[1.0, 2.0])
except ValueError as error:
    print("잡아낸 ValueError:", error)

# [2단계] 다섯 조각 모두에 대해 BM25 점수(없으면 0)와 코사인 점수를 같은 순서로 모읍니다
SPREAD_QUERY = "점심 먹는 곳 여는 시간"
bm25_all = dict(index.search(SPREAD_QUERY, limit=5))
bm25_list = []
for chunk in chunks:
    bm25_list.append(bm25_all.get(chunk.chunk_id, 0.0))
bm25_values = np.array(bm25_list)
cosine_values = vectors @ np.array(embeddings.embed_query(SPREAD_QUERY))

# [3단계] 두 점수의 폭(최대-최소)을 비교하고, 정규화 없이 더한 순위가 어느 쪽을 따라가는지 봅니다
print("BM25 점수 폭 (최대-최소)  :", round(float(bm25_values.max() - bm25_values.min()), 3))
print("코사인 점수 폭 (최대-최소):", round(float(cosine_values.max() - cosine_values.min()), 3))

def ids_in_order(scores, kind="quicksort"):
    """점수 배열을 높은 순서로 정렬해 조각 ID 목록으로 돌려줍니다."""
    return [chunks[i].chunk_id for i in np.argsort(-scores, kind=kind)]

summed = bm25_values + cosine_values
print("단순 합산 순위  :", ids_in_order(summed))
print("BM25 단독 순위  :", ids_in_order(bm25_values, kind="stable"))
print("코사인 단독 순위:", ids_in_order(cosine_values))

정규화 없이 두 원점수를 그냥 더하면, 점수 변동 폭(`최대 - 최소`)이 훨씬 큰 쪽이 합산 순위를 사실상 독점해 버립니다(위 출력에서는 BM25 점수가 있는 `s0`가 코사인 점수 차이를 압도해 1위를 결정합니다). 따라서 두 검색기를 결합할 때는 RRF처럼 순위를 쓰거나, 뒤에서 배울 `fuse_scores`(`rag_lab/retrieval.py:104`)처럼 두 점수를 먼저 `0.0 ~ 1.0` 구간으로 정규화한 뒤 가중 합산해야 합니다.

## 6. 청킹 규칙, 메타데이터는 자격 (2분)

이제 실제 사내 업무 문서(`data/business/`)를 불러와 문서를 조각내는 **청킹(Chunking)** 규칙을 살펴보겠습니다. 우리 수업의 기본 청킹 함수인 `section_chunks`(`rag_lab/corpus.py:105`)는 마크다운의 `## ` 절 제목을 기준으로 나누며 **절 안에 있는 마크다운 표는 절대로 중간에서 자르지 않습니다.** 비교 대상인 `fixed_chunks`(`rag_lab/corpus.py:88`)는 문서 구조를 보지 않고 지정한 글자 수(`size`)와 겹침(`overlap`)만으로 기계적으로 자릅니다.

한편 문서에 붙어 있는 고객(`entity`), 유효 기간(`valid_from` ~ `valid_until`), 승인 상태(`status`) 같은 **메타데이터는 점수를 가감할 '관련도'가 아니라 검색 후보가 될 수 있는 '자격 조건'**입니다. 따라서 범위 필터는 검색 점수를 매기기 **전에** 자격 없는 문서를 먼저 제외해야 합니다.

In [ ]:
# [따라하기] 절 단위 청킹(section_chunks)과 고정 글자 수 청킹(fixed_chunks)의 조각 경계를 비교합니다 (무료)

# [1단계] 문서를 불러오고 서비스 계약서 v2 하나를 고릅니다
documents = load_corpus()
v2 = documents["sla-a-v2"]

# [2단계] 같은 문서를 두 가지 규칙으로 자릅니다
by_section = section_chunks(v2)
by_size = fixed_chunks(v2, size=400, overlap=80)

# [3단계] 규칙별 조각 수, 중간에 잘린 표의 수, 조각마다 시작과 끝 글자를 출력합니다
for name, pieces in (("절 단위 청킹", by_section), ("고정 길이(400/80)", by_size)):
    cut_cnt = count_cut_tables(v2, pieces)
    print(f"[{name}] 총 조각 {len(pieces)}개, 중간에 잘린 표 {cut_cnt}개")
    for c in pieces:
        beginning = c.text[:24]
        ending = c.text[-18:]
        print(f"  {c.chunk_id:14s} | 시작={beginning!r} ... 끝={ending!r}")

절 단위 조각(`section_chunks`)은 항상 `## ` 제목으로 시작해 문단이나 표가 끝나는 곳에서 깔끔하게 끝납니다. 반면 고정 길이 조각(`fixed_chunks`)은 문장이나 단어 한가운데서 잘립니다. 위 `400/80` 설정에서 잘린 표가 `0개`로 나온 것은 그저 표 길이가 400자 안에 우연히 들어갔기 때문일 뿐입니다.

아래 `[바꿔보기]` 셀에서 고정 길이 청킹의 조각 크기 `SIZE`를 `150`으로 줄여 보겠습니다(겹침 `OVERLAP`은 `30`). **먼저 예측해 보세요:** 중간에 잘린 표(`count_cut_tables`)가 몇 개로 나올까요?

In [ ]:
# [바꿔보기] 바꿀 값: SIZE와 OVERLAP. 예측을 먼저 적고 실행하세요. (무료)
SIZE, OVERLAP = 150, 30

small = fixed_chunks(v2, size=SIZE, overlap=OVERLAP)
cut_tables = count_cut_tables(v2, small)
print(f"총 조각 {len(small)}개, 중간에 잘린 표 {cut_tables}개")

조각 크기를 150자로 줄이자 표가 중간에 잘렸습니다. 표가 잘리면 구체적으로 어떤 문제가 생기는지, 아래 `[깨뜨리기]` 셀을 실행해 열 머리글(header)을 잃어버린 표 조각의 실제 내용을 직접 눈으로 확인해 보세요.

In [ ]:
# [깨뜨리기] 150/30 고정 길이 조각 중에서 열 머리글(header)을 잃어버린 표 조각을 찾습니다 (무료)
header = "| 월간 가용률 | 서비스 크레딧 | 계산 기준 |"

for c in small:
    is_after_start = c.start > 0      # 문서 맨 앞 조각이 아니고
    has_table_row = "|" in c.text     # 표의 행이 들어 있는데
    lost_header = header not in c.text  # 열 머리글은 없는 조각을 찾습니다
    if is_after_start and has_table_row and lost_header:
        print(c.chunk_id, "->", repr(c.text))

표가 중간에서 잘려 열 머리글(`| 월간 가용률 | 서비스 크레딧 | 계산 기준 |`)이 앞 조각에 남고 뒤 조각에는 `| 99.0% 미만 | 20% | ... |` 같은 행만 남으면, 그 조각만 읽는 모델은 **`20%`가 도대체 무엇의 비율인지** 알 수 없습니다. 표를 온전히 지키는 것은 글자 수 조정이 아니라 청킹 규칙입니다.

## 7. 범위 필터 (3분)

`Retriever.rank(질문, scope)`(`rag_lab/retrieval.py:143`)는 검색 점수를 계산하기 전에 `Document.applies(scope)`(`rag_lab/corpus.py:41`)를 먼저 확인하여 조건에 맞지 않는 문서 조각을 아예 검색 대상에서 **제외**합니다. 자격 조건은 세 가지입니다.
1. 문서 상태가 승인됨(`status == "approved"`)이어야 합니다(초안 `draft` 제외).
2. 문서의 고객이 질의 범위의 고객(`scope.entity`)과 같거나 `"공통"` 문서여야 합니다.
3. 기준일(`scope.as_of`)이 문서의 유효 기간(`valid_from <= as_of < valid_until`) 안에 있어야 합니다.

In [ ]:
# [따라하기] 범위 필터 없이 검색할 때와 범위 필터(Scope)를 걸고 검색할 때의 결과 문서를 비교합니다 (무료)

# [1단계] BM25 검색기를 만들고 질문을 정합니다
retriever_bm25 = Retriever(documents)
QUESTION = "알파 서비스 크레딧 신청 기한은 언제인가요?"

def to_docs(chunk_ids):
    """조각 ID 목록을 문서 ID 목록으로 바꾸되, 원래 순서를 유지하며 중복만 제거합니다."""
    doc_ids = []
    for chunk_id in chunk_ids:
        doc_ids.append(retriever_bm25.chunks[chunk_id].doc_id)
    return list(dict.fromkeys(doc_ids))

# [2단계] 범위 필터 없이(None) 검색한 결과와 알파·기준일 범위를 걸고 검색한 결과를 얻습니다
without_scope = to_docs(retriever_bm25.rank(QUESTION, None, limit=6))
scope = Scope("알파", date(2026, 9, 8))
within_scope = to_docs(retriever_bm25.rank(QUESTION, scope, limit=6))

# [3단계] 두 결과와, 필터가 걸러 낸 자격 미달 문서를 출력합니다
filtered_out = []
for doc_id in without_scope:
    if doc_id not in within_scope:
        filtered_out.append(doc_id)
print("범위 필터 없음:", without_scope)
print("범위 필터 적용:", within_scope)
print("필터로 걸러진 자격 미달 문서:", filtered_out)

범위 필터를 켜자 미승인 초안(`sla-a-draft`), 다른 고객의 계약서(`sla-b`), 그리고 2026년 9월 8일 기준으로 이미 만료된 구버전(`sla-a-v1`)이 후보에서 깔끔하게 사라졌습니다. 관련도가 낮아서가 아니라 **응답에 쓰일 자격이 없어서** 검색 전에 제외된 것입니다.

이번에는 아래 `[바꿔보기]` 셀에서 기준일 `AS_OF`를 `2026년 6월 30일`로 되돌려 보겠습니다. **먼저 예측해 보세요:** `sla-a-v1`, `sla-a-v2`, `sla-a-amendment`, `sla-a-draft` 중 어느 문서가 자격을 얻어 남을까요?

In [ ]:
# [바꿔보기] 바꿀 값: AS_OF. 예측을 먼저 적고 실행하세요. (무료)
AS_OF = date(2026, 6, 30)

# [1단계] 기준일을 바꾼 범위로 검색해 남은 문서를 봅니다
old_scope = Scope("알파", AS_OF)
remaining_docs = to_docs(retriever_bm25.rank(QUESTION, old_scope, limit=6))
print("기준일 변경 후 남은 문서:", remaining_docs)

# [2단계] 알파 계약서 계열 문서마다 유효 기간, 상태, 새 기준일에서의 자격을 출력합니다
for doc_id in ("sla-a-v1", "sla-a-v2", "sla-a-amendment", "sla-a-draft"):
    doc = documents[doc_id]
    until = doc.valid_until or "현재"
    qualified = doc.applies(old_scope)
    print(f"  {doc_id:16s} | 유효기간: {doc.valid_from} ~ {until:10s} | 상태: {doc.status:8s} | 자격(applies)={qualified}")

기준일이 2026년 6월 30일로 바뀌자, 7월 1일부터 발효되는 `sla-a-v2`와 `sla-a-amendment`가 빠지고 그 시점에 유효했던 구버전 계약서 `sla-a-v1`만 알파 계약서 계열 가운데 남았습니다(출력의 `travel`은 계약서 계열이 아닌 다른 문서입니다).

그렇다면 왜 범위 필터를 검색 **전에(사전 필터, Pre-filtering)** 걸어야 하고, 일단 상위 `k`개를 뽑은 **뒤에(사후 필터, Post-filtering)** 걸면 안 될까요? 아래 `[깨뜨리기]` 셀을 실행해 사후 필터와 사전 필터의 남은 후보 수를 비교하고, 임베딩 없이 Dense 검색기를 만들었을 때의 예외도 함께 확인해 보세요.

In [ ]:
# [깨뜨리기] 검색 후 거르는 사후 필터(Post-filtering)와 검색 전 거르는 사전 필터(Pre-filtering)를 비교하고, 임베딩 없는 dense 생성을 시도합니다 (무료)

# [1단계] 사후 필터: 범위 없이 상위 2개 문서를 먼저 뽑은 뒤에 자격을 검사합니다
top2_unfiltered = to_docs(retriever_bm25.rank(QUESTION, None, limit=2))
post_filtered = []
for doc_id in top2_unfiltered:
    if documents[doc_id].applies(scope):
        post_filtered.append(doc_id)

# [2단계] 사전 필터: 자격 있는 조각 중에서 상위 2개 문서를 뽑습니다
pre_filtered = to_docs(retriever_bm25.rank(QUESTION, scope, limit=2))

print("상위 2개를 먼저 뽑은 뒤 범위로 거름(사후 필터):", post_filtered, f"-> 남은 후보 {len(post_filtered)}개")
print("범위로 먼저 거른 뒤 상위 2개를 뽑음(사전 필터):", pre_filtered, f"-> 남은 후보 {len(pre_filtered)}개")

# [3단계] 임베딩 없이 dense 검색기를 만들면 조용히 BM25로 대체하지 않고 ValueError가 납니다
try:
    Retriever(documents, mode="dense")
except ValueError as error:
    print("잡아낸 ValueError:", error)

점수 상위 2개를 먼저 뽑은 뒤에 자격 조건을 검사하는 **사후 필터(Post-filtering)**를 쓰면, 자격 없는 문서(`sla-a-draft` 등)가 상위 자리를 차지했다가 뒤늦게 탈락하면서 정작 필요한 정답 문서가 밀려나 후보 수가 모자라게 됩니다. 그래서 자격 범위(`Scope`)는 반드시 점수를 매기기 전에 걸어야 합니다. 또한 `Retriever`에 임베딩 모델을 넘기지 않고 `mode="dense"`를 요청하면 조용히 BM25로 대체하지 않고 즉시 `ValueError`를 내어 설정 실수를 알려 줍니다.

## 8. 측정: recall, 누락과 오인 (2분)

검색 성능을 숫자로 평가할 때는 세 가지를 구분해서 봅니다(`rag_lab/evaluation.py:19`).
- **재현율(Recall@k)**: 정답 문서(`expected_docs`) 가운데 검색 상위 `k`개 문서 안에 들어온 비율입니다.
- **누락(`missing_docs`)**: 정답 문서인데 상위 `k`개 안에서 찾지 못한 문서입니다.
- **오인(`out_of_scope_docs`)**: 현재 질문의 범위(`Scope`) 밖인데 상위 `k`개 안에 잘못 딸려 들어온 자격 미달 문서입니다.

> **주의**: 아래 측정에 사용하는 두 정답표(`data/gold_hard.jsonl`, `data/gold_hard2.jsonl`)는 아직 사람의 최종 검수를 거치지 않은 **검토 전 초안(`review_status: draft`)**입니다. 만약 정답표에 아쉬운 행이 보이더라도, 측정 결과를 본 뒤에 점수를 높이려고 정답표를 사후에 고치지 않습니다.

## 9. 어려운 질문으로 측정 (4분)

이제 세 가지 기본 검색 방식(`bm25`, `dense`, `hybrid`)을 어려운 질문 두 모음(`cases_hard.jsonl` 14문항 + `cases_hard2.jsonl` 22문항 = 총 36문항)에서 측정해 비교합니다. 평가 전용 정답표(`gold_hard*.jsonl`)는 Agent 실행 경로에서는 절대 읽지 않으며, 오직 아래 측정 셀에서만 읽습니다.

In [ ]:
# [따라하기] 어려운 질문 두 모음(36문항)에서 질문 유형별 recall@2를 측정합니다 (무료, 모델 호출 0회)

# [1단계] 어려운 질문 두 모음(14문항 + 22문항 = 36문항)과 검토 전 초안 정답표를 불러옵니다
cases_1 = load_cases(ROOT / "data/cases_hard.jsonl")
cases_2 = load_cases(ROOT / "data/cases_hard2.jsonl")
cases = cases_1 + cases_2  # 두 질문 모음의 ID는 서로 겹치지 않습니다

gold = {}
gold.update(load_gold(ROOT / "data/gold_hard.jsonl"))
gold.update(load_gold(ROOT / "data/gold_hard2.jsonl"))

# 질문 ID가 어느 모음(hard, hard2)에서 왔는지 기록해 둡니다
SET_OF = {}
for case_row in cases_1:
    SET_OF[case_row["id"]] = "hard"
for case_row in cases_2:
    SET_OF[case_row["id"]] = "hard2"

# [2단계] 비교할 세 가지 기본 검색기(bm25, dense, hybrid)를 준비합니다
retrievers = {}
for mode in ("bm25", "dense", "hybrid"):
    retrievers[mode] = Retriever(documents, embeddings=embeddings, mode=mode)

results = {}  # (검색 방식, k) -> 문항별 평가 결과 리스트. 아래 셀들에서 이어서 사용합니다

# [3단계] 지정한 k에 대해 재현율(recall)을 측정하는 함수와 결과 출력 함수를 정의합니다
def measure(k):
    """지정한 k에 대해 세 가지 검색 방식의 평가 결과를 results 사전에 저장합니다."""
    for mode, retriever in retrievers.items():
        results[(mode, k)] = evaluate_retrieval(retriever, cases, gold, k=k)

def show(k):
    """지정한 k에 대한 질문 유형별 평균 recall 표와 실패 행 상세 내역을 출력합니다."""
    # 방식별로 {질문 유형: 요약 행} 사전을 만듭니다
    summaries = {}
    for mode in retrievers:
        by_type = {}
        for item in recall_by_type(results[(mode, k)]):
            by_type[item["type"]] = item
        summaries[mode] = by_type

    header = ""
    for mode in retrievers:
        header += f"{mode:>9}"
    print(f"k={k}   {header}")

    for case_type in summaries["bm25"]:
        formatted = ""
        for mode in retrievers:
            value = summaries[mode][case_type]["recall"]
            text = "-" if value is None else format(value, ".2f")
            formatted += f"{text:>9}"
        n = summaries["bm25"][case_type]["cases"]
        print(f"{case_type:14}{formatted}   (질문 {n}개)")

    # 정답 문서를 놓쳤거나(누락) 범위 밖 문서를 가져온(오인) 행은 모두 출력합니다
    print()
    for mode in retrievers:
        for row in results[(mode, k)]:
            if row["missing_docs"] or row["out_of_scope_docs"]:
                print(
                    f"  실패 [{mode:6s}] {row['case_id']}: "
                    f"찾음={row['retrieved_docs']} | 누락={row['missing_docs']} | 오인={row['out_of_scope_docs']}"
                )

# [4단계] 상위 2개 문서 기준(k=2)으로 측정하고 결과를 확인합니다
measure(2)
show(2)

이번에는 상위 문서 수 `k`를 `1, 2, 3`으로 바꾸어 가며 측정하고, 순위 품질 지표인 **MRR(Mean Reciprocal Rank)**을 함께 계산해 전체 표로 보겠습니다. MRR은 첫 번째로 등장한 정답 문서 순위의 역수(1위면 `1.0`, 2위면 `0.5`, 3위면 `0.333...`, 상위 `k`개 밖이면 `0.0`)를 모든 채점 문항에 대해 평균 낸 값이며, 아래 표에서는 `k=3` 기준으로 계산합니다.

In [ ]:
# [따라하기] 두 질문 모음(hard, hard2)과 합계에서 recall@1,2,3 및 MRR@3을 방식별로 집계합니다 (무료, 모델 호출 0회)
from rag_lab.evaluation import mrr

# [1단계] 상위 문서 수 k=1, 2, 3에 대해 각각 검색 평가를 실행합니다
for k in (1, 2, 3):
    measure(k)

# [2단계] 질문 모음별(hard, hard2, 전체) 부분집합 추출 및 평균 recall 계산 함수를 정의합니다
def subset(rows, name):
    """전체 행에서 지정한 질문 모음('hard', 'hard2', '전체')에 속하는 행만 추려냅니다."""
    if name == "전체":
        return rows
    picked = []
    for row in rows:
        if SET_OF[row["case_id"]] == name:
            picked.append(row)
    return picked

def mean_recall(rows):
    """정답 문서가 있는 문항(recall이 None이 아닌 행)들의 평균 recall과 채점 문항 수를 돌려줍니다."""
    values = []
    for row in rows:
        if row["recall"] is not None:
            values.append(row["recall"])
    mean_val = (sum(values) / len(values)) if values else None
    return mean_val, len(values)

def fmt(value):
    """숫자는 소수 셋째 자리까지, 값이 없으면 '-'로 표시합니다."""
    return "-" if value is None else f"{value:.3f}"

# [3단계] 질문 모음별로 recall@1, recall@2, recall@3, MRR@3 비교 표를 출력합니다
for name in ("hard", "hard2", "전체"):
    n_all = len(subset(results[("bm25", 1)], name))
    n_scored = mean_recall(subset(results[("bm25", 1)], name))[1]
    print(f"[{name}] 질문 {n_all}개 (정답 문서가 있어 채점한 질문 {n_scored}개)")
    print(f"{'방식':8}{'recall@1':>10}{'recall@2':>10}{'recall@3':>10}{'MRR@3':>9}{'누락 행(k=2)':>14}")
    for mode in retrievers:
        recall_cols = ""
        for k in (1, 2, 3):
            recall_value = mean_recall(subset(results[(mode, k)], name))[0]
            recall_cols += f"{fmt(recall_value):>10}"
        mrr_val = mrr(subset(results[(mode, 3)], name))[0]
        failed = 0
        for row in subset(results[(mode, 2)], name):
            if row["missing_docs"]:
                failed += 1
        print(f"{mode:8}{recall_cols}{fmt(mrr_val):>9}{failed:>14}")
    print()

# [4단계] k=2에서 정답 문서를 놓친 모든 실패 행을 숨김없이 출력합니다
print("누락이 발생한 모든 행 (k=2 기준, 숨기거나 지운 행 없음):")
for mode in retrievers:
    for row in results[(mode, 2)]:
        if row["missing_docs"]:
            print(
                f"  [{mode:6s}] {row['case_id']} ({SET_OF[row['case_id']]}): "
                f"찾음={row['retrieved_docs']} | 누락={row['missing_docs']}"
            )

위 집계 표에서 `hard` 모음과 `hard2` 모음의 지표별 1위 방식이 서로 같은지 비교해 보세요. 아래 결론 정리 셀은 정답표를 다시 읽지 않고 방금 위에서 측정한 `results` 값만 그대로 읽어, 각 질문 모음과 지표별 1위 방식이 어떻게 갈리는지 요약해 보여 줍니다.

In [ ]:
# 결론 정리 셀 (표지 없음): 위에서 측정한 results 값만 읽어 요약합니다. 정답표를 다시 읽지 않습니다.

# [1단계] 비교할 4가지 평가지표(recall@1, recall@2, recall@3, MRR@3)를 이름 있는 함수로 정의합니다
def recall_at_1(mode, name):
    """질문 모음 name에서 검색 방식 mode의 recall@1 평균을 돌려줍니다."""
    return mean_recall(subset(results[(mode, 1)], name))[0]

def recall_at_2(mode, name):
    """질문 모음 name에서 검색 방식 mode의 recall@2 평균을 돌려줍니다."""
    return mean_recall(subset(results[(mode, 2)], name))[0]

def recall_at_3(mode, name):
    """질문 모음 name에서 검색 방식 mode의 recall@3 평균을 돌려줍니다."""
    return mean_recall(subset(results[(mode, 3)], name))[0]

def mrr_at_3(mode, name):
    """질문 모음 name에서 검색 방식 mode의 MRR@3을 돌려줍니다."""
    return mrr(subset(results[(mode, 3)], name))[0]

KEYS = [
    ("recall@1", recall_at_1),
    ("recall@2", recall_at_2),
    ("recall@3", recall_at_3),
    ("MRR@3", mrr_at_3),
]

# [2단계] 질문 모음별(hard, hard2, 전체)로 각 지표의 1위 검색 방식을 판별합니다
winners_by_set = {}
for name in ("hard", "hard2", "전체"):
    total_cnt = len(subset(results[("bm25", 1)], name))
    scored_cnt = mean_recall(subset(results[("bm25", 1)], name))[1]
    print(f"[{name}] 전체 {total_cnt}개 중 채점 {scored_cnt}개 (정답표는 검토 전 초안입니다)")
    for label, metric in KEYS:
        # 방식별 지표 값을 구하고, 소수점 아홉째 자리까지 같으면 동점으로 봅니다
        values = {}
        for mode in retrievers:
            values[mode] = metric(mode, name)
        best = max(round(v, 9) for v in values.values())
        top = []
        for mode, value in values.items():
            if round(value, 9) == best:
                top.append(mode)
        winners_by_set[(name, label)] = tuple(top)

        if len(top) == len(values):
            verdict = "세 방식 동점"
        elif len(top) > 1:
            verdict = f"공동 1위(동점) {top}"
        else:
            verdict = f"1위 {top[0]}"
        rounded_scores = {}
        for mode, value in values.items():
            rounded_scores[mode] = round(value, 3)
        print(f"  {label:9s} -> {verdict:26s} {rounded_scores}")

# [3단계] 질문 모음에 따라 1위 방식이 바뀐 지표가 있는지 확인하고 최종 결론을 출력합니다
disagree = []
for label, _ in KEYS:
    if winners_by_set[("hard", label)] != winners_by_set[("hard2", label)]:
        disagree.append(label)
print("\n두 질문 모음(hard vs hard2)에서 1위 방식이 달라진 지표:", disagree or "없음")

missing_per_mode = {}
for mode in retrievers:
    missing_rows = 0
    for row in results[(mode, 2)]:
        if row["missing_docs"]:
            missing_rows += 1
    missing_per_mode[mode] = missing_rows
print("방식별 누락 행 수 (k=2):", missing_per_mode)

out_of_scope_total = 0
for rows in results.values():
    for row in rows:
        out_of_scope_total += len(row["out_of_scope_docs"])
print("범위 밖 문서(오인) 총합:", out_of_scope_total)

# 전체 표본의 지표별 1위 방식이 모두 같은 세 방식인지(동점), 하나의 같은 조합인지 확인합니다
all_tied = True
overall_winner_sets = set()
for label, _ in KEYS:
    overall_winners = winners_by_set[("전체", label)]
    overall_winner_sets.add(overall_winners)
    if len(overall_winners) != len(retrievers):
        all_tied = False

if all_tied:
    print("결론: 전체 표본에서는 모든 지표에서 세 방식이 동점이라 우열을 가려내지 못합니다.")
elif len(overall_winner_sets) == 1:
    print(
        "결론: 전체 표본에서는 모든 지표의 1위가 같지만:",
        list(winners_by_set[("전체", "recall@1")]),
        "문항 수가 적고 정답표가 초안이므로 문항 한두 개가 순위를 바꿀 수 있습니다.",
    )
else:
    print("결론: 전체 표본에서 지표별로 1위 방식이 갈립니다. 문항 수가 적고 정답표가 초안이므로 문항 한두 개가 순위를 바꿀 수 있습니다.")

### 찾은 뒤 읽기 (`related_docs`와 `read_more`)

단일 검색 호출(`search_knowledge`)에서 상위 1개만 가져왔을 때 연관된 추가 약정 문서를 놓치더라도, 우리 검색 도구는 찾은 문서와 같은 문서군(`family`)에 속한 범위 안 문서를 `related_docs` 목록으로 함께 알려 줍니다(`rag_lab/evidence.py:146`). Agent는 이 목록을 보고 `read_evidence`(`read_more`) 도구로 연관 문서를 이어서 읽어 볼 수 있습니다.

In [ ]:
# [따라하기] 상위 1개만 검색한 뒤 related_docs를 따라가 read_more로 연관 문서를 읽습니다 (무료)

# [1단계] 어려운 질문 모음에서 질문 하나를 고르고, 그 질문의 고객과 기준일로 범위(Scope)를 만듭니다
for candidate in cases:
    if candidate["id"] == "para-claim-window":
        case = candidate
        break
case_scope_obj = Scope(case["entity"], date.fromisoformat(case["as_of"]))

# [2단계] hybrid 검색기를 감싼 검색 도구(RetrievalContext)로 상위 1개만 검색합니다
context = RetrievalContext(retrievers["hybrid"], case_scope_obj)
found = context.search(case["question"], limit=1)
found_doc_ids = []
for evidence in found["evidence"]:
    found_doc_ids.append(evidence["doc_id"])
print("첫 검색으로 찾은 문서:", found_doc_ids)
print("함께 안내된 연관 문서(related_docs):", found["related_docs"])

# [3단계] 안내된 연관 문서 중 첫 번째를 read_more로 이어서 읽고 앞부분 미리보기를 출력합니다
for doc_id in found["related_docs"][:1]:
    more = context.read_more(doc_id)
    quote = more["evidence"]["quote"]
    preview = quote[:80].replace("\n", " / ")
    print(f"이어서 읽은 문서({doc_id}) 미리보기:", preview)

상위 1개 검색(`limit=1`)에서는 본계약 문서만 나왔지만, `related_docs`에 추가 약정 문서가 안내되었고 이를 `read_more`로 읽자 변경된 기한 조항이 나왔습니다. 이처럼 Agent가 2단계로 **'찾은 뒤 이어서 읽는' 동작은 단순 1회 검색의 문서 재현율(recall@1) 표에는 드러나지 않습니다.**

## 10. 넓은 후보와 원 질문 재순위 (7분)

검색 뒤에 교차 인코더 재순위(Reranker)를 붙일 때 가장 중요한 원칙은 **"초기 후보 깊이(pool)가 최종 재현율(recall)의 천장(상한선)이다"**라는 점입니다. 재순위 모델은 1차 검색기가 뽑아 준 후보 목록 **안에서만** 순서를 다시 줄 세울 뿐이므로, 1차 후보 컷(`pool`) 밖으로 밀려난 정답 조각은 재순위 모델이 아무리 뛰어나도 되살릴 수 없습니다.

- 기존의 `hybrid` 방식은 BM25와 Dense 각각 상위 18개에서 자른 뒤 RRF로 합칩니다(`rag_lab/retrieval.py:16`).
- 새로 도입한 `fusion` 방식은 범위 안의 모든 조각에 대해 정규화된 점수를 구하고, Dense 가중치 `0.7`로 볼록 결합해 후보를 넓게 만듭니다(`rag_lab/retrieval.py:104`).
- 실제 PDF 벤치마크인 올거나이즈(Allganize) `hold` 분할(1회 측정)에서 RRF 후보 18개(`MRR@10 0.692`)를 점수 융합 + 재순위 후보 50개로 넓혔을 때 `MRR@10`이 `0.766`으로 올랐습니다(`docs/retrieval_benchmarks.md:13`).

이 파이프라인 구조는 오픈소스 [AutoRAG](https://github.com/Marker-Inc-Korea/AutoRAG)의 커밋 `0c39052`(v2.13.4) 설계를 참고했습니다. 아래 `src/agent/agent.ts`, `src/retrieval/rerank.ts` 경로는 AutoRAG 저장소(커밋 0c39052)의 경로이며 이 저장소에는 없습니다. 이 저장소에서 찾으려고 하지 마세요.
1. 질문을 하위 질의로 분해하고 각 질의로 검색합니다.
2. 질의별 검색 결과를 번갈아 섞어(`interleave`) 합칩니다(`src/agent/agent.ts:3120`).
3. 합쳐진 후보들을 분해된 질의가 아니라 **원래 질문(original question)** 기준으로 재순위합니다(`src/agent/agent.ts:3145`).
4. 상위 25개(`src/retrieval/rerank.ts:42`)를 최종 후보로 남깁니다.

이 저장소에서는 `retrieve` 함수(`rag_lab/pipeline.py:63`) 하나가 이 과정을 수행합니다. 아래 셀에서는 수업 문서 전체 문항(기본 12문항 + 어려운 질문 36문항 = 48문항)으로 `hybrid`와 `fusion`을 비교해 봅니다. 정답 문서 자체가 질문의 고객·기준일 범위 밖에 있는 함정 문항은 어떤 검색기도 찾을 수 없으므로, 지우지 않고 `범위 안` 집계 열에서만 따로 분리해 보여 줍니다.

먼저 48문항 전체를 돌리기 전에, 앞선 다섯 문장 예제에서 순위만 합치는 `rrf()`와 정규화된 점수를 `0.7 : 0.3`으로 합치는 `fuse_scores()`가 어떻게 동작하는지 한눈에 비교해 봅시다.


In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 다섯 문장에서 RRF(순위 합산)와 fuse_scores(정규화 점수 결합) 비교하기
from rag_lab.retrieval import fuse_scores

demo_q = "점심 먹는 곳 여는 시간"

# [1단계] BM25 원점수 목록과 Dense 원점수 목록을 (조각 ID, 점수) 쌍으로 만듭니다
sparse_pairs = index.search(demo_q, limit=5)
dense_vals = vectors @ np.array(embeddings.embed_query(demo_q))
dense_pairs = []
for i in np.argsort(-dense_vals):
    dense_pairs.append((chunks[i].chunk_id, float(dense_vals[i])))

# [2단계] 원점수를 보기 좋게 반올림해 출력합니다
sparse_rounded = []
for chunk_id, score in sparse_pairs:
    sparse_rounded.append((chunk_id, round(score, 3)))
dense_rounded = []
for chunk_id, score in dense_pairs:
    dense_rounded.append((chunk_id, round(score, 3)))
print("BM25 원점수  :", sparse_rounded)
print("Dense 원점수 :", dense_rounded)

# [3단계] 순위만 쓰는 RRF와 정규화한 점수를 0.7 : 0.3으로 합치는 fuse_scores의 결과를 비교합니다
sparse_ids = [chunk_id for chunk_id, _ in sparse_pairs]
dense_ids = [chunk_id for chunk_id, _ in dense_pairs]
print("RRF 순위 결합        :", rrf([sparse_ids, dense_ids]))
print("fuse_scores 점수 결합:", fuse_scores(dense_pairs, sparse_pairs, dense_weight=0.7))

**관찰할 포인트:**
- `fuse_scores`는 Dense 점수를 `최소~최대(0~1)`로, BM25 점수를 `최댓값 기준(0~1)`으로 먼저 정규화한 뒤 `0.7 * Dense + 0.3 * BM25`로 합산합니다. 덕분에 원점수의 단위 차이 문제를 피하면서도, 순위(1위·2위·3위)만 볼 때는 버려지던 **점수 간격 정보**를 살릴 수 있습니다.

이제 수업 문서 전체 48문항(기본 12문항 + 어려운 질문 36문항)에서 `hybrid`(RRF)와 `fusion`(`fuse_scores` + 출처 보강)을 같은 표로 측정해 비교해 봅시다.


In [ ]:
# [따라하기] 수업 문서 전체 48문항에서 hybrid와 fusion을 같은 표로 측정해 비교합니다 (무료, 모델 호출 0회)

# [1단계] 전체 48문항(기본 12 + 어려운 질문 36)과 점수 융합(fusion) 검색기를 준비합니다
all_cases = load_cases() + cases  # 기본 12문항 + 어려운 질문 두 모음 36문항 = 총 48문항
all_gold = {**load_gold(), **gold}
fusion = Retriever(documents, embeddings=embeddings, mode="fusion")

def find_case(case_list, case_id):
    """질문 목록에서 id가 case_id인 질문 한 개를 찾아 돌려줍니다. 없으면 StopIteration이 납니다."""
    return next(c for c in case_list if c["id"] == case_id)

def top_docs(run, k=3):
    """RetrievalRun 결과의 조각 ID들을 문서 ID로 바꾸어 상위 k개를 돌려줍니다."""
    doc_ids = []
    for chunk_id in run.chunk_ids:
        doc_ids.append(fusion.chunks[chunk_id].doc_id)
    return list(dict.fromkeys(doc_ids))[:k]

def case_scope(case):
    """질문 한 개의 고객(entity)과 기준일(as_of)로 검색 범위(Scope)를 만듭니다."""
    return Scope(case["entity"], date.fromisoformat(case["as_of"]))

# [2단계] 정답 문서가 없는 보류 문항(NO_EXPECTED)과 정답 문서가 범위 밖인 문항(OUT_OF_SCOPE)을 파악합니다
NO_EXPECTED = []
for one_case in all_cases:
    if not all_gold[one_case["id"]]["expected_docs"]:
        NO_EXPECTED.append(one_case["id"])

OUT_OF_SCOPE = []
for one_case in all_cases:
    case_id = one_case["id"]
    if case_id in NO_EXPECTED:
        continue
    # 정답 문서가 모두 이 질문의 범위에서 자격이 있어야 '범위 안' 문항입니다
    expected_docs = all_gold[case_id]["expected_docs"]
    all_qualified = True
    for doc_id in expected_docs:
        if not documents[doc_id].applies(case_scope(one_case)):
            all_qualified = False
    if not all_qualified:
        OUT_OF_SCOPE.append(case_id)

# [3단계] 전체(45문항)와 범위 안(41문항)을 나누어 측정하는 함수와 순위 변동 비교 함수를 정의합니다
def measure_all(name, retriever, rank=None):
    """48문항에 대해 전체(45문항)와 범위 안(41문항)으로 나누어 recall@1,2,3과 MRR@3을 출력합니다."""
    rows = {}
    for k in (1, 2, 3):
        rows[k] = evaluate_retrieval(retriever, all_cases, all_gold, k=k, rank=rank)

    for label, excluded in (("전체", []), ("범위 안", OUT_OF_SCOPE)):
        # 제외할 문항(범위 밖)을 뺀 행만 남깁니다
        kept = {}
        for k in rows:
            kept_rows = []
            for row in rows[k]:
                if row["case_id"] not in excluded:
                    kept_rows.append(row)
            kept[k] = kept_rows
        recalls = []
        for k in (1, 2, 3):
            recalls.append(fmt(mean_recall(kept[k])[0]))
        mean_rr, scored = mrr(kept[3])
        print(
            f"{name:12s} [{label:4s}] "
            f"r@1={recalls[0]}  r@2={recalls[1]}  r@3={recalls[2]}  "
            f"MRR@3={fmt(mean_rr)}  (채점 {scored}문항)"
        )
    return rows

def compare(before, after, k=3):
    """두 검색 결과(before -> after) 사이에서 정답 문서 순위가 좋아진 행과 나빠진 행을 모두 출력합니다."""
    # 정답 문서가 있는 행마다 역순위(MRR)가 얼마나 변했는지 계산합니다
    gains = []
    for before_row, after_row in zip(before[k], after[k]):
        if not before_row["expected_docs"]:
            continue
        change = mrr([after_row])[0] - mrr([before_row])[0]
        gains.append((after_row["case_id"], change, before_row["retrieved_docs"], after_row["retrieved_docs"]))

    improved = [g for g in gains if g[1] > 0]
    worsened = [g for g in gains if g[1] < 0]
    for title, picked in (("순위가 좋아진 행", improved), ("순위가 나빠진 행", worsened)):
        print(f"{title}: {len(picked)}개")
        for case_id, change, was, now in picked:
            print(f"  {case_id}: 역순위 변화 {change:+.2f}  ({was} -> {now})")

    same_cnt = 0
    for g in gains:
        if g[1] == 0:
            same_cnt += 1
    print(f"순위가 그대로인 행: {same_cnt}개")

# [4단계] 말뭉치 조각 수와 문항 구성을 확인한 뒤 hybrid와 fusion을 측정해 비교합니다
sizes = []
for one_case in all_cases:
    in_scope_chunks = fusion.rank(one_case["question"], case_scope(one_case), limit=len(fusion.chunks))
    sizes.append(len(in_scope_chunks))
average_size = sum(sizes) / len(sizes)
print(f"전체 조각 {len(fusion.chunks)}개 | 문항별 범위 안 조각 수: 최소 {min(sizes)}개 ~ 최대 {max(sizes)}개 (평균 {average_size:.1f}개)")
print(f"전체 {len(all_cases)}문항 = 채점 대상 {len(all_cases) - len(NO_EXPECTED)}문항 + 정답 문서 없음 {len(NO_EXPECTED)}문항 {NO_EXPECTED}")
print(f"정답 문서가 범위 밖인 문항 {len(OUT_OF_SCOPE)}개: {OUT_OF_SCOPE}\n")

hybrid_rows = measure_all("hybrid", retrievers["hybrid"])
fusion_rows = measure_all("fusion", fusion)
print()
compare(hybrid_rows, fusion_rows)

우리 수업용 말뭉치는 전체 조각이 32개뿐이고, 고객·기준일 범위 필터를 거치면 문항당 후보 조각이 출력된 개수(평균 18개 안팎)만큼만 남습니다. 따라서 이 작은 말뭉치에서는 후보 컷 18개(`hybrid`)와 50개(`fusion`)의 깊이 차이가 거의 드러나지 않습니다. 후보 깊이의 차이는 수천 개의 조각이 있는 대규모 말뭉치에서 본격적으로 나타납니다.

이제 로컬 교차 인코더(Cross-Encoder) 모델인 `make_reranker`(`BAAI/bge-reranker-v2-m3`)를 붙여 보겠습니다. 임베딩(Bi-Encoder)이 질문과 문서를 따로 벡터로 만든 뒤 내적만 계산한다면, 교차 인코더는 **(질문, 후보 조각) 한 쌍을 모델에 함께 넣어** 두 문장 사이의 세밀한 상호작용을 직접 읽고 관련도 점수를 매깁니다(`rag_lab/rerank.py:29`). **먼저 예측해 보세요:** 재순위를 붙이면 `r@1`(1위 적중률)과 `r@3`(상위 3개 재현율) 중 어느 쪽이 더 크게 오를까요?

In [ ]:
# [따라하기] 교차 인코더 재순위(fusion+재순위) 전후 성능을 같은 48문항에서 비교합니다 (무료, 로컬 모델, 첫 실행은 내려받기로 느림)

# [1단계] 로컬 교차 인코더 모델을 준비하고 지정한 후보 깊이(pool)로 재순위하는 함수를 정의합니다
reranker = make_reranker()
warnings_seen = []

def make_rank(pool):
    """지정한 후보 수(pool)로 재순위하는 검색 함수를 만듭니다.
    evaluate_retrieval이 k=1,2,3마다 반복 호출하므로 캐시(memo)를 두어 질문당 한 번만 재순위합니다."""
    memo = {}

    def rank(question, scope):
        key = (question, scope)
        if key not in memo:
            run = retrieve(question, retriever=fusion, scope=scope, reranker=reranker, pool=pool)
            warnings_seen.extend(run.warnings)
            memo[key] = run.chunk_ids
        return memo[key]

    return rank

# [2단계] 후보 50개에 대해 재순위(fusion+재순위)를 수행하고 재현율과 MRR을 측정합니다
started = perf_counter()
rerank_rows = measure_all("fusion+재순위", fusion, make_rank(50))
elapsed_per_q = (perf_counter() - started) / len(all_cases)
print(f"소요 시간: 문항당 평균 {elapsed_per_q:.2f}초 | 재순위 경고: {len(warnings_seen)}건")

# [3단계] 재순위 전(fusion)과 후(fusion+재순위)의 순위 변동을 비교하고 놓친 행을 모두 출력합니다
print("\n[재순위 전후 비교: fusion -> fusion+재순위]")
compare(fusion_rows, rerank_rows)

print("\n상위 3개에서도 놓친 행 (k=3, 숨기거나 지운 행 없음):")
for row in rerank_rows[3]:
    if row["missing_docs"]:
        if row["case_id"] in OUT_OF_SCOPE:
            reason = "범위 밖: 어느 구성으로도 찾을 수 없음"
        else:
            reason = "범위 안 검색 실패"
        print(f"  [{reason}] {row['case_id']}: 찾음={row['retrieved_docs']} | 누락={row['missing_docs']}")

print("\n1위에서 놓친 행 (k=1):")
for row in rerank_rows[1]:
    if row["missing_docs"]:
        print(f"  {row['case_id']}: 찾음={row['retrieved_docs']} | 누락={row['missing_docs']}")

재순위 모델은 1차 검색이 이미 찾아 놓은 후보들 중에서 진짜 정답 조각을 맨 위(1위)로 끌어올리는 역할을 합니다. 그 결과 `r@1`과 `MRR@3`은 뚜렷하게 오르지만, 이미 `1.000`이었던 범위 안 `r@3`은 그대로입니다(범위 밖 문항 4개는 `전체` 열에 0점으로 그대로 포함되어 있습니다).

아래 `[바꿔보기]` 셀에서는 재순위에 넘기는 1차 후보 수 `POOLS`를 `1, 2, 5, 18`로 바꾸어 후보 깊이가 만드는 천장을 직접 눈으로 확인해 봅시다. **먼저 예측해 보세요:** 재순위에 넘기는 후보가 단 `1개`뿐이면 범위 안 `r@3`은 얼마가 될까요? 그리고 후보가 몇 개 이상일 때부터 점수가 더 이상 오르지 않고 멈출까요?

In [ ]:
# [바꿔보기] 바꿀 값: POOLS (재순위 모델에 넘길 1차 후보 조각 수). 예측을 먼저 적고 실행하세요. (무료, 약 1분 소요)
POOLS = [1, 2, 5, 18]

# 후보 수(pool)를 바꿔 가며 같은 48문항을 재측정합니다
for pool in POOLS:
    rank_with_pool = make_rank(pool)
    measure_all(f"후보 {pool}개", fusion, rank_with_pool)
print("비교 참고: 후보 50개 결과는 바로 위 셀의 'fusion+재순위' 표와 같습니다.")

위 표를 보면 후보가 1개일 때는 `r@3`이 1차 검색의 `r@1`을 절대 넘지 못하고, 후보 수를 늘릴수록 천장이 올라가다가 이 말뭉치에서는 후보 5개부터 후보 18개와 점수가 같아지는 것을 확인할 수 있습니다(후보 2개는 범위 안 `r@3`이 0.963으로 아직 5개보다 낮습니다).

그렇다면 운영 중에 만약 교차 인코더 재순위 모델이 메모리 부족이나 타임아웃으로 오류를 내면 검색 전체가 멈춰야 할까요? 아래 `[깨뜨리기]` 셀을 실행해 재순위 모델이 고장 났을 때 `retrieve`가 어떻게 안전하게 대처하는지 확인해 보세요.

In [ ]:
# [깨뜨리기] 재순위 모델(CrossEncoderReranker) 적재/실행 중 오류가 날 때 검색 파이프라인이 어떻게 대처하는지 확인합니다 (무료)
from rag_lab.rerank import CrossEncoderReranker

# [1단계] 존재하지 않는 로컬 모델 경로를 지정해 실제 CrossEncoderReranker에서 OSError가 발생하는 상황을 만듭니다
unloaded_reranker = CrossEncoderReranker("/nonexistent-local-reranker-path")

# [2단계] 이 재순위 모델로 검색하면 어떤 경고가 남고 결과 순서가 어떻게 되는지 봅니다
probe = find_case(all_cases, "table-ratio-max")
probe_scope = case_scope(probe)
broken = retrieve(probe["question"], retriever=fusion, scope=probe_scope, reranker=unloaded_reranker)
first_stage_order = fusion.rank(probe["question"], probe_scope, limit=25)

print("기록된 경고(warnings):", broken.warnings)
print("1차 융합 검색 순서를 그대로 유지했는가:", broken.chunk_ids == first_stage_order)

재순위 모델에서 예외가 발생하더라도 `retrieve` 함수는 검색 전체를 중단시키지 않고, 1차 융합 검색(`fusion`)이 만든 순서를 그대로 돌려주며 `warnings`에 `rerank-failed: <예외 이름>` 경고를 남깁니다(바로 위 셀에서는 존재하지 않는 모델 경로를 줬으므로 `rerank-failed: OSError`가 기록되었습니다)(`rag_lab/pipeline.py:79`). AutoRAG 역시 재순위가 실패하면 원래 순서를 유지하도록 설계되어 있습니다(`src/agent/agent.ts:3182`, AutoRAG 저장소(커밋 0c39052)의 경로이며 이 저장소에는 없습니다).

## 11. Jev 관문: 분해와 출처 보강은 기본으로 켠다 (6분)

**질문 분해(Decomposition)를 두는 이유:** `"장애 대응 절차와 감사 로그 보관 기간을 함께 알려 주세요"`처럼 **서로 다른 두 문서의 사실을 하나로 묶은 복합 질문**은, 원 질문 그대로 한 번만 검색하면 어느 한쪽 문서의 조각이 상위권에서 밀려날 수 있습니다. 그래서 AutoRAG는 질문을 여러 하위 질의로 쪼개 각각 검색한 뒤 결과를 번갈아 합칩니다(`interleave`).

우리 수업에서도 질문 분해를 **기본값으로 켜 두되**, 모든 질문마다 비싼 LLM을 불러 쪼개지 않고 **Decision API(Jev) 관문**에게 먼저 물어봅니다. `needs_decomposition`(`rag_lab/gates.py:93`)이 반환한 점수 `P`가 문턱값 `0.5`(`rag_lab/gates.py:18`) 이상일 때만 Claude를 불러 `decompose_question`을 실행합니다. 관문을 호출해 열렸는지 판정하는 줄이 `rag_lab/phases.py:54`이고, 관문이 열렸을 때 `decompose_question`을 호출하는 줄이 `rag_lab/phases.py:57`입니다.

**Decision API(Jev)의 점수 `P`를 읽을 때 주의할 점:**
- 반환된 `P`는 통계적으로 보정된(calibrated) 실제 확률이 아닙니다(`rag_lab/decisions.py:3`).
- 문턱값 `0.5` 역시 데이터로 학습해 찾은 최적값이 아니라 코드에 정해 둔 규칙이며, 공급자마다 점수 분포와 보정 특성이 다릅니다.
- **관문을 호출할 때 질문 문장 원문이 선택한 Decision API 공급자에게 그대로 전송됩니다.**
- 만약 API 호출이 실패하면 검색을 멈추지 않고 '분해하지 않음(`False`)'을 기본값으로 사용합니다.

In [ ]:
# [따라하기] 수업 문서 질문 세 개를 분해 관문(needs_decomposition)에 넣어 판단 결과를 확인합니다 (Jev 3회, Claude 0회)
decider = make_decider()
GATE_IDS = ["cross-ops-and-log-retention", "simple-deadline-current", "table-ratio-max"]

print(f"[분해 관문] Decision API(Jev) 호출 {len(GATE_IDS)}회, Claude 호출 0회입니다. 질문 문장이 공급자로 전송됩니다. 진행합니다.")
for case_id in GATE_IDS:
    question = find_case(all_cases, case_id)["question"]
    gate = needs_decomposition(decider, question)

    # 관문이 열렸는지(분해 실행)와, API 실패로 대체 기본값을 썼는지를 읽기 쉬운 글로 바꿉니다
    decision_text = "예(열림)" if gate.decision else "아니요(닫힘)"
    fallback_note = f" (대체 기본값 사용: {gate.fallback})" if gate.fallback else ""
    print(
        f"{case_id:28s} | backend={gate.backend} model={gate.model} "
        f"P={gate.probability} 문턱값={gate.threshold} "
        f"-> 분해 여부: {decision_text}{fallback_note}"
    )

저자가 이전에 수업 문서 48문항 전체를 1회 훑어 측정했을 때는 총 5문항(비율 `0.104`)에서만 관문이 열렸고, `cross-ops-and-log-retention`이 그 다섯 문항 중 하나였습니다(`docs/retrieval_benchmarks.md:153`). 방금 실행한 출력에서 세 질문의 `P`와 열림·닫힘을 이 관찰과 나란히 비교해 보세요. 관문은 모델 응답이라 같은 질문도 실행마다 `P`가 달라질 수 있고(같은 질문을 11절 아래 `[바꿔보기]` 셀에서 다시 물으면 앞 셀과 다른 `P`가 나올 수 있습니다), 문턱값 `0.5` 근처에서는 결과가 뒤집힐 수 있습니다. 저자의 실행(1회 관찰)에서는 복합 질문인 `cross-ops-and-log-retention`이 `0.72`로 열렸고, 단일 질문인 `simple-deadline-current`(`0.14`)와 `table-ratio-max`(`0.22`)는 닫혔습니다. 이 점수 `P`는 호출할 때마다 조금씩 달라질 수 있습니다.

참고로 Decision API 공급자별 공식 문서는 `docs.typesafe.ai`와 `developers.openai.com/api/docs/guides/decisions`에서 볼 수 있습니다. 우리 저장소의 `rag_lab/decisions.py`가 각 공급자의 요청·응답 형식을 어떻게 맞추어 두었는지 요약하면 아래 표와 같습니다.

| 공급자 백엔드 (`rag_lab/decisions.py`) | 참/거짓 질문 필드 | 판단 기준 문장 필드 | 응답 형식 |
|---|---|---|---|
| `TypeSafeBackend`, `OpenRouterBackend` | `noul` | `criteria` | 질문 이름을 키로 하는 딕셔너리(map) |
| `OpenAIDecisionsBackend` | `predicate` | `instructions`에 합침 | 판정 객체 리스트(list) |

새로운 Decision API 공급자를 추가하고 싶다면 `_HttpBackend`를 상속받아 요청 본문(`_body`)과 응답 파싱(`_verdicts`) 메서드만 구현한 뒤 `BACKENDS`(`rag_lab/decisions.py:172`)에 등록하면 됩니다. `make_decider`(`rag_lab/decisions.py:175`)는 `.env`에 키가 설정된 첫 번째 공급자를 자동으로 선택하며, 키가 하나도 없으면 안내 메시지와 함께 멈춥니다.

이제 아래 `[바꿔보기]` 셀에서 관문이 열린 질문을 실제로 분해해 검색 후보를 합쳐 보겠습니다. **먼저 예측해 보세요:** 질문을 분해해 후보를 합치면 최종 상위 문서 목록이 달라질까요? 그리고 `QUESTION_ID`를 관문이 닫히는 `"simple-deadline-current"`로 바꾸면 어떻게 동작할까요?

In [ ]:
# [바꿔보기] 바꿀 값: QUESTION_ID. 관문이 열리면 질문을 분해해 후보를 합치고 원 질문으로 재순위합니다 (Jev 1회, 열리면 Claude 1회)
QUESTION_ID = "cross-ops-and-log-retention"

# [1단계] 재순위 모델에 어떤 질문 문장이 들어갔는지 기록하는 래퍼(RecordingReranker)를 정의합니다
class RecordingReranker:
    """재순위 모델에 어떤 질문 문장이 들어왔는지 기록만 하고, 실제 점수 계산은 원래 reranker에 맡깁니다."""
    def __init__(self, inner):
        self.inner = inner
        self.queries = set()

    def score(self, query, texts):
        self.queries.add(query)
        return self.inner.score(query, texts)

# [2단계] Decision API 분해 관문(needs_decomposition)으로 질문 분해 필요 여부를 먼저 판정합니다
probe = find_case(all_cases, QUESTION_ID)
probe_scope = case_scope(probe)
print("[분해 관문] Decision API(Jev) 호출 1회, Claude 호출은 관문이 열릴 때만 1회입니다. 진행합니다.")
gate = needs_decomposition(decider, probe["question"])
gate_text = "열림(분해 실행)" if gate.decision else "닫힘(분해 생략)"
print(f"관문 판정: P={gate.probability}, 문턱값={gate.threshold} -> {gate_text}")

# [3단계] 관문이 열렸을 때(gate.decision == True)만 Claude를 호출해 하위 질의로 분해합니다
split_queries = []
if gate.decision:
    announce("질문 분해", model_calls=1)
    split_queries = decompose_question(make_model(), probe["question"])

# [4단계] 원 질문 단독 검색(plain)과 하위 질의 합침 검색(split)을 같은 재순위 모델로 실행합니다
recorder = RecordingReranker(reranker)
plain = retrieve(probe["question"], retriever=fusion, scope=probe_scope, reranker=recorder)
split = retrieve(
    probe["question"],
    retriever=fusion,
    scope=probe_scope,
    reranker=recorder,
    queries=split_queries or None,
)

# [5단계] 후보 수와 상위 문서를 비교하고, 재순위 모델에 들어간 질의가 원 질문뿐인지 확인합니다
print("원래 질문:", probe["question"], "| 정답 문서:", all_gold[QUESTION_ID]["expected_docs"])
print("분해된 하위 질의:", split_queries or "(분해하지 않음)")
allowed = len(fusion.rank(probe["question"], probe_scope, limit=len(fusion.chunks)))
print(
    f"검색에 쓴 질의 총 {len(split.queries)}개 -> 합쳐진 후보 {split.candidates}개 "
    f"(전체 조각 {len(fusion.chunks)}개 중 현재 Scope가 허용한 조각 {allowed}개)"
)
print(f"원 질문만 검색 : 후보 {plain.candidates}개 -> 상위 문서 {top_docs(plain)}")
print(f"분해 후 합침   : 후보 {split.candidates}개 -> 상위 문서 {top_docs(split)}")
print("재순위 모델에 들어간 질의가 오직 '원 질문' 하나뿐인가:", recorder.queries == {probe["question"]})

### 출처 보강 (`corroborate`)

**출처 보강**(`corroborate`, `rag_lab/retrieval.py:111`)은 같은 출처 문서에서 여러 조각이 뽑혔거나 Dense와 BM25 양쪽에서 동시에 찾은 조각에 가산점(상한 `0.3`)을 주는 규칙이며, `Retriever`의 기본값(`corroboration=True`)으로 켜져 있습니다.

전체 문항을 비교하기 전에, 조각 세 개짜리 작은 예시로 `corroborate()` 함수가 어떻게 순위를 뒤집는지 먼저 확인해 봅시다.


In [ ]:
# [따라하기] 무료, 모델 호출 0회 — 출처 보강(corroborate)이 같은 문서·복수 검색기 조각에 가산점을 주는 원리 확인하기
from rag_lab.retrieval import corroborate

# c1은 기본 점수(0.75)가 가장 높지만 단독 문서(doc-X)에서 dense 하나로만 찾은 조각입니다.
# c2(0.70)와 c3(0.60)는 같은 문서(doc-Y)에서 나왔고 dense와 bm25 양쪽에서 함께 찾은 조각입니다.
sample_scores = {"c1": 0.75, "c2": 0.70, "c3": 0.60}
sample_sources = {"c1": "doc-X", "c2": "doc-Y", "c3": "doc-Y"}
sample_methods = {"c1": {"dense"}, "c2": {"dense", "bm25"}, "c3": {"dense", "bm25"}}

def by_score_desc(chunk_id):
    """정렬 기준: 기본 점수가 높은 조각이 먼저 오도록 점수에 음수를 붙입니다."""
    return -sample_scores[chunk_id]

print("보강 전 점수 순위:", sorted(sample_scores, key=by_score_desc))
print("보강 후 최종 순위:", corroborate(sample_scores, sample_sources, sample_methods))

**관찰할 포인트:**
- 기본 점수는 `c1(0.75)`이 `c2(0.70)`보다 높았지만, `c2`는 같은 출처 문서(`doc-Y`)의 다른 조각(`c3`)과 함께 뽑혔고 두 검색기(`dense`, `bm25`) 모두에서 등장해 가산점을 받아 1위로 올라섰습니다.

이제 실제 사내 문서에서 이 출처 보강을 켰을 때와 껐을 때의 1차 융합(`fusion`) 순서를 비교해 봅니다(앞서 설명한 구조적 이유 때문에 교차 인코더 재순위를 붙이면 수업 문서에서는 차이가 사라지므로, 재순위 없이 1차 융합 순서만 비교합니다). **먼저 예측해 보세요:** 가산점을 켜고 끄면 실제 질문에서 조각 순위가 바뀔까요?


In [ ]:
# [바꿔보기] 바꿀 값: CORRO_ID. 예측을 먼저 적고 실행하세요. 재순위 없이 출처 보강(corroboration) 켬/끔의 1차 융합 순서를 비교합니다 (무료)
CORRO_ID = "simple-amendment-scope"

# [1단계] 출처 보강을 끈 fusion 검색기를 만들고, 보강을 켠 fusion과 같은 질문으로 상위 8개 조각을 얻습니다
fusion_off = Retriever(documents, embeddings=embeddings, mode="fusion", corroboration=False)
probe = find_case(all_cases, CORRO_ID)
probe_scope = case_scope(probe)
on_ids = fusion.rank(probe["question"], probe_scope, limit=8)
off_ids = fusion_off.rank(probe["question"], probe_scope, limit=8)

# [2단계] 보강을 켠 순서대로 조각마다 보강을 껐을 때의 순위를 나란히 출력합니다
print("질문:", probe["question"])
for rank, chunk_id in enumerate(on_ids, 1):
    off_rank = off_ids.index(chunk_id) + 1 if chunk_id in off_ids else "-"
    moved = "   <- 순위 변동" if off_rank != rank else ""
    print(f"  보강 켬 {rank}위: {chunk_id:30s} (보강 끔 {off_rank}위){moved}")

# [3단계] 상위 3개 문서 순서를 비교합니다
def first_three_docs(chunk_ids):
    """조각 ID 목록을 문서 ID로 바꾸고 중복을 없앤 뒤 앞의 세 개만 돌려줍니다."""
    doc_ids = []
    for chunk_id in chunk_ids:
        doc_ids.append(fusion.chunks[chunk_id].doc_id)
    return list(dict.fromkeys(doc_ids))[:3]

print("상위 3개 문서 순서 (보강 켬):", first_three_docs(on_ids), "| (보강 끔):", first_three_docs(off_ids))

# [4단계] 48문항 전체에서 보강 켬/끔에 따라 1차 융합 순서가 달라진 문항을 셉니다
differ = []
for one_case in all_cases:
    one_scope = case_scope(one_case)
    order_on = fusion.rank(one_case["question"], one_scope, limit=len(fusion.chunks))
    order_off = fusion_off.rank(one_case["question"], one_scope, limit=len(fusion.chunks))
    if order_on != order_off:
        differ.append(one_case["id"])
print(f"재순위 없는 1차 융합 순서가 보강 켬/끔에 따라 달라진 문항: {len(differ)}개 / 전체 {len(all_cases)}개")

저자의 실행에서는 상위 조각 8개 중 7개의 순위가 바뀌었고, 전체 48문항 중 46문항에서 1차 융합 순서가 달라졌습니다. 하지만 여기에 교차 인코더 재순위(`pool=50`)를 붙이면 수업 문서 48문항에서는 문서 순위 차이가 0건이 됩니다(범위 안 조각이 모두 50개 안에 들어가기 때문입니다).

그래서 조각 수가 1,992개인 올거나이즈 `tune` 분할에서 출처 보강의 실제 효과를 측정해 보았습니다(`docs/retrieval_benchmarks.md:339`).

| MRR@10 변화 (보강 끔 -> 보강 켬) | 교차 인코더 재순위가 **정상**일 때 | 교차 인코더 재순위가 **실패**했을 때 |
|---|---|---|
| 출처 단위를 **쪽(page)**으로 묶음 | `+0.008` (`-0.005 ~ +0.029`) | `-0.057` (`-0.090 ~ -0.029`) |
| 출처 단위를 **파일(file)**로 묶음 | `+0.009` (`-0.001 ~ +0.029`) | `-0.015` (`-0.039 ~ -0.001`) |

측정 결과를 정직하게 읽으면 다음과 같습니다.
- 재순위가 정상 동작할 때는 95% 구간이 `0`을 걸쳐 있어 측정 가능한 유의미한 변화가 없습니다.
- 오히려 재순위 모델이 실패해 1차 융합 순서가 그대로 노출될 때는, 출처 보강을 켠 쪽의 성능이 뚜렷하게 낮아졌습니다(출처를 파일 단위로 묶으면 하락 폭이 약 4분의 1로 줄어들지만 여전히 구간 전체가 음수입니다).
- 즉, 우리 벤치마크 실험에서는 **출처 보강이 검색 성능을 높여 준다는 수치적 근거가 나오지 않았습니다.** 그럼에도 기본값을 `True`로 켜 둔 것은 AutoRAG(`src/retrieval/merger.ts:22`, AutoRAG 저장소(커밋 0c39052)의 경로이며 이 저장소에는 없습니다) 설계를 그대로 따르기로 한 저자의 설계 선택이며, 왜 재순위 실패 시 점수가 낮아지는지 그 원인은 아직 추가 분석하지 않았습니다.

## 12. AutoRAG와 다른 점 (3분)

우리 수업 코드가 오픈소스 AutoRAG에서 그대로 가져온 것과 다르게 선택한 것을 측정 수치와 함께 정리하면 다음과 같습니다. 아래 표와 설명에 나오는 `src/retrieval/merger.ts`, `src/retrieval/types.ts`, `docs/minsync-setup.md` 경로는 AutoRAG 저장소(커밋 0c39052)의 경로이며 이 저장소에는 없습니다.

| 비교 항목 | AutoRAG 설계 | 이 수업(`rag_lab`)의 선택 | 측정 수치 (저자 1회 실행 기준) |
|---|---|---|---|
| **토크나이저** | 공백 단어 + 형태소 결합 (`docs/minsync-setup.md:287`) | 형태소(`tokenize`)만 사용 (`rag_lab/retrieval.py:25`) | Ko-StrategyQA BM25 단독 nDCG@10: 형태소만 `0.581` vs 단어+형태소 `0.558` |
| **출처 보강** | 같은 출처 문서 조각에 가산점 (`src/retrieval/merger.ts:22`) | 기본값으로 켬 (`Retriever` 기본값) | 재순위 정상 시 변화 없고, 재순위 실패 시 하락 (위 11절 표 참고) |
| **범위 필터** | 파일 경로 접두사 범위만 지원 (`src/retrieval/types.ts:33`) | 고객(`entity`)·기준일(`as_of`)·승인 상태 사전 필터 | 아래 `[깨뜨리기]` 셀에서 직접 측정 |

여기서 가장 눈여겨볼 점은 **"한 벤치마크의 1위 구성이 실제 업무용 PDF 문서로 그대로 이어지지 않았다"**는 사실입니다.
- 분해된 각 하위 질의마다 재순위 점수를 매겨 최고 점수를 취하는 방식은 위키백과 문단 모음인 Ko-StrategyQA에서 1위(`nDCG@10 0.835`)를 차지했지만, 실제 PDF 문서 시험인 올거나이즈 `tune`에서는 오히려 1위 적중률이 떨어졌습니다(`0.714` 대 원 질문 재순위 `0.762`). 그래서 우리는 AutoRAG와 똑같이 **원 질문 기준 재순위**를 채택했습니다.
- 형태소 단독 토크나이저 역시 Ko-StrategyQA 한 곳에서만 이긴 결과이므로 다른 말뭉치에서는 순위가 뒤집힐 위험이 있습니다(실제로 AutoRAG 자체 문서인 `docs/minsync-setup.md:300`에는 단어+형태소 결합이 더 나았다는 반대 방향의 실험값이 적혀 있습니다).

아래 `[깨뜨리기]` 셀을 실행해 만약 우리 수업 코드에서 고객·기준일 사전 범위 필터를 꺼 버리면 전체 48문항 중 몇 문항에서 자격 미달 문서(오인)가 올라오는지 확인해 보세요.

In [ ]:
# [깨뜨리기] 고객·기준일 범위 필터(use_scope)를 껐을 때 범위 밖 자격 미달 문서가 얼마나 섞여 올라오는지 세어 봅니다 (무료)
for label, use_scope in (("범위 필터 켬", True), ("범위 필터 끔", False)):
    scope_rows = evaluate_retrieval(fusion, all_cases, all_gold, k=3, use_scope=use_scope)

    # 상위 3개 문서 안에 범위 밖 문서(오인)가 섞인 문항만 모읍니다
    wrong = []
    for row in scope_rows:
        if row["out_of_scope_docs"]:
            wrong.append(row)
    wrong_ids = [row["case_id"] for row in wrong]

    print(f"[{label}] 상위 3개 문서 안에 범위 밖 문서(오인)가 섞인 문항: {len(wrong)}개 / 전체 {len(scope_rows)}개")
    print("  오인이 발생한 문항 ID:", wrong_ids)

범위 필터를 끄면 자격 미달 문서가 섞여 들어오는 오인(out-of-scope)이 대거 발생합니다(2026-11-02 실행 기준 `0문항 -> 47문항`, 전체 48문항 중). 교차 인코더 재순위 모델은 문장의 의미적 관련도만 볼 뿐 그 문서가 어느 고객의 계약서인지, 유효 기간이 지났는지는 판별하지 못하므로 범위 필터를 대신할 수 없습니다.

## 13. PDF 쪽 판독 (5분)

실제 기업용 PDF 벤치마크(올거나이즈)에서 **가장 큰 성능 향상을 만든 부품**은 검색 공식 변경이 아니라 바로 이 **PDF 쪽 판독(Page Reading)**이었습니다. 보고서의 차트 이미지나 스캔된 페이지에는 PDF 내부에 텍스트 레이어(글자층)가 아예 없어서, 아무리 정교한 검색기와 재순위 모델을 써도 **색인할 글자 자체가 존재하지 않기 때문**입니다.

실습용 표본 파일인 `data/pdf/credit-trend-chart.pdf`는 글자층 없이 차트 이미지만 들어 있는 한 쪽짜리 PDF이며, 차트 안의 수치는 수업용으로 만든 가상 예시입니다. 먼저 일반 PDF 텍스트 추출기로 읽었을 때 글자층과 검색 결과가 어떻게 나오는지 확인해 보겠습니다.

In [ ]:
# [따라하기] 글자층이 없는 차트 PDF 쪽은 일반 텍스트 추출과 검색에 전혀 잡히지 않음을 확인합니다 (무료)
PDF_PATH = ROOT / "data/pdf/credit-trend-chart.pdf"
PAGE_QUERY = "서비스 크레딧 신청 건수가 가장 많은 분기"

def page_search(text):
    """쪽 하나의 글자 text를 조각 하나로 만들어 PAGE_QUERY로 BM25 검색한 결과를 돌려줍니다."""
    chunk = Chunk("p1", "credit-trend-chart", "", text, 0, len(text), 1, 1)
    return BM25Index([chunk]).search(PAGE_QUERY)

# [1단계] PDF 첫 쪽에서 일반 텍스트 추출기(pymupdf)로 글자층을 읽고, 판독이 필요한 쪽인지 확인합니다
with pymupdf.open(PDF_PATH) as pdf:
    layer = pdf[0].get_text()
    page_needs_reading = needs_reading(pdf[0])
    print("전체 쪽 수:", pdf.page_count, "| 추출된 글자층:", repr(layer), "| 판독 필요 여부(needs_reading):", page_needs_reading)

# [2단계] 추출한 글자층으로 검색해 봅니다. 글자가 없으면 검색 결과도 비어 있습니다
print("추출된 글자층으로 검색한 결과:", page_search(layer))

추출된 글자층이 빈 문자열(`''`)이라 검색 결과도 비어 있고, `needs_reading`이 `True`(판독 필요)를 돌려주었습니다.

이제 아래 셀을 실행해 멀티모달 모델(Claude)로 이 PDF의 1쪽 이미지를 읽어 검색 가능한 텍스트로 옮긴 뒤, 똑같은 질의로 다시 검색해 보겠습니다.

In [ ]:
# [따라하기] 멀티모달 모델(Claude)이 PDF 쪽 이미지를 읽어 검색 가능한 글자로 옮깁니다 (유료, 모델 호출 1회)

# [1단계] 모델 호출 1회를 고지한 뒤 PDF 1쪽을 이미지로 읽어 텍스트로 옮깁니다
announce("PDF 1쪽 이미지를 모델로 판독해 텍스트로 옮깁니다", model_calls=1)
transcript = read_page(make_model(), PDF_PATH, 1)

# [2단계] 판독한 텍스트와, 그 텍스트로 같은 질의를 검색한 결과를 출력합니다
print("[모델이 판독해 옮긴 텍스트]")
print(transcript)
print("\n판독 텍스트로 검색한 결과:", page_search(transcript))

출력에서 먼저 확인할 것은 두 가지입니다. 모델이 판독해 옮긴 글자에 차트 제목과 분기별 수치가 들어 있는지, 그리고 그 글자로 `page_search`를 했을 때 글자층이 없던 앞 셀(`[]`)과 달리 검색 결과(점수)가 생겼는지입니다. 판독 문장의 표현은 실행마다 달라질 수 있으므로 문장이 아니라 이 두 가지를 보세요. 옮겨 적은 수치가 원본 차트와 같은지는 코드가 확인해 주지 않습니다.

참고로 `read_page`에 존재하지 않는 쪽 번호(예: 1쪽짜리 문서에서 2쪽)를 넣으면 어떻게 될까요? 아래 `[깨뜨리기]` 셀을 실행해 불필요한 유료 모델 호출이 일어나기 전에 코드가 미리 쪽 번호 범위를 검사해 막아 주는지 확인해 보세요.

In [ ]:
# [깨뜨리기] 존재하지 않는 쪽 번호(2쪽)를 요청하면 모델을 호출하기 전에 즉시 예외로 막는지 확인합니다 (무료)
try:
    read_page(make_model(), PDF_PATH, 2)  # 1쪽짜리 문서에 2쪽을 요청합니다
except ValueError as error:
    print("잡아낸 ValueError:", error)

이처럼 `needs_reading`(`rag_lab/pages.py:38`)은 이미지 면적이 쪽의 15% 이상이거나 추출된 글자가 100자 미만인 쪽을 골라내고, `read_page`(`rag_lab/pages.py:44`)가 그 쪽을 이미지로 렌더링해 멀티모달 모델로 글자를 읽어 옵니다. 판독해 낸 텍스트를 색인에 넣자 비로소 검색에서 후보가 잡힙니다.

단, 여기서도 중요한 운영 원칙이 있습니다. **모델이 차트나 표 이미지를 읽어 옮긴 글자의 수치 정확성은 아직 코드로 검증되지 않았습니다.** 검색 후보를 찾아내는 데는 큰 도움이 되지만, 최종 답변에서 그 수치를 인용하려면 원본 PDF 쪽 이미지와 대조하는 절차가 따로 필요합니다.

참고로 저자가 올거나이즈 `hold` 분할에서 전체 1,008쪽 중 `needs_reading` 조건에 걸린 309쪽만 골라 판독했을 때(비용 `$3.89`, 소요 시간 5분 26초), 전체 `MRR@10`은 `0.766`에서 `0.873`으로 뛰었고, 이미지 유형 문항의 상위 5위 적중은 `14/24`에서 `21/24`로 크게 올랐습니다(`docs/retrieval_benchmarks.md:15`).

## 직접 해보기 (5분)

아래 세 개의 `[내 것 만들기]` 셀에는 **예시 문장과 질문이 미리 채워져** 있어서 그대로 실행해 볼 수 있습니다. 먼저 실행해 결과를 확인한 뒤, `MY_`로 시작하는 변수들을 내가 다루는 업무 문장과 질문으로 바꾸어 다시 실행해 보세요.

먼저 **첫 번째 실습**으로, 내가 만든 다섯 문장(`MY_SENTENCES`)과 질의(`MY_QUERY`)에 대해 형태소 토큰화, BM25, 코사인 유사도, RRF 합산 순위를 한 번에 비교해 봅니다.

In [ ]:
# [내 것 만들기] 내 문장 다섯 개와 내 질의로 토큰화, BM25, 코사인 유사도, RRF 합산을 한 번에 비교합니다 (무료)
MY_SENTENCES = [
    "팀 회의는 매주 화요일 오전 10시에 합니다.",
    "회의록은 회의 당일 저녁까지 공유 폴더에 올립니다.",
    "점심 회식은 팀장 승인을 받은 뒤에 예약합니다.",
    "휴가는 사흘 전까지 캘린더에 먼저 등록합니다.",
    "긴급 장애는 당직 담당자에게 전화로 먼저 알립니다.",
]
MY_QUERY = "회식 하려면 누구한테 허락받아요?"

if len(MY_SENTENCES) < 2 or not MY_QUERY.strip():
    print("MY_SENTENCES에 문장을 두 개 이상, MY_QUERY에 검색 질의를 채워 주세요.")
else:
    # [1단계] 내 문장을 조각으로 만들고 BM25 색인을 만든 뒤 문장마다 토큰을 출력합니다
    mine = []
    for i, sentence in enumerate(MY_SENTENCES):
        mine.append(Chunk(f"m{i}", "mine", "", sentence, 0, len(sentence), 1, 1))
    my_index = BM25Index(mine)
    for i, sentence in enumerate(MY_SENTENCES):
        print(f"m{i} 토큰:", tokenize(sentence))

    # [2단계] BM25 결과와 코사인 유사도 결과(점수가 높은 순)를 각각 구합니다
    bm25_hits = my_index.search(MY_QUERY, limit=len(mine))
    doc_vecs = np.array(embeddings.embed_documents(MY_SENTENCES))
    q_vec = np.array(embeddings.embed_query(MY_QUERY))
    cos = doc_vecs @ q_vec
    dense_order = np.argsort(-cos)
    dense_hits = [mine[i].chunk_id for i in dense_order]

    print("\nBM25 결과  :", bm25_hits)
    cosine_pairs = []
    for i in dense_order:
        cosine_pairs.append((mine[i].chunk_id, round(float(cos[i]), 3)))
    print("코사인 결과:", cosine_pairs)

    # [3단계] 두 순위 목록을 RRF로 합쳐 상위 3개를 문장과 함께 출력합니다
    bm25_ids = [chunk_id for chunk_id, _ in bm25_hits]
    fused = rrf([bm25_ids, dense_hits])
    top3 = []
    for chunk_id in fused[:3]:
        top3.append((chunk_id, MY_SENTENCES[int(chunk_id[1:])]))
    print("RRF 상위 3 :", top3)

이어서 **두 번째 실습**에서는 수업 문서(`data/business/`)를 대상으로 내 질문(`MY_QUESTION`)을 던졌을 때, 네 가지 검색 방식(`bm25`, `dense`, `hybrid`, `fusion+재순위`)이 내가 기대한 정답 문서(`MY_EXPECTED_DOC`)를 상위 2개 안에서 찾아내는지 비교해 보세요.

In [ ]:
# [내 것 만들기] 내 질문이 기대한 문서를 네 가지 검색 방식(bm25, dense, hybrid, fusion+재순위)의 상위 2개 안에서 찾는지 비교합니다 (무료)
MY_QUESTION = "출장 가서 하룻밤 자면 숙박비를 얼마까지 낼 수 있나요?"
MY_EXPECTED_DOC = "travel"
MY_ENTITY = "공통"
MY_AS_OF = "2026-09-08"

if not MY_QUESTION.strip() or MY_EXPECTED_DOC not in documents:
    print("MY_QUESTION을 채우고, MY_EXPECTED_DOC은 다음 문서 ID 중 하나로 지정하세요:", list(documents))
else:
    my_scope = Scope(MY_ENTITY, date.fromisoformat(MY_AS_OF))

    # [1단계] 기본 세 검색기(bm25, dense, hybrid)로 순위를 얻고 상위 2개 문서 안에 기대 문서가 있는지 봅니다
    for mode, retriever in retrievers.items():
        ids = retriever.rank(MY_QUESTION, my_scope, limit=len(retriever.chunks))
        ranked_docs = []
        for chunk_id in ids:
            ranked_docs.append(retriever.chunks[chunk_id].doc_id)
        top2 = list(dict.fromkeys(ranked_docs))[:2]
        verdict = "찾음" if MY_EXPECTED_DOC in top2 else "못 찾음"
        print(f"{mode:7s} | 상위 2개 문서={top2} -> 기대 문서({MY_EXPECTED_DOC}): {verdict}")

    # [2단계] fusion 검색에 교차 인코더 재순위를 붙인 결과도 같은 방법으로 확인합니다
    reranked_run = retrieve(MY_QUESTION, retriever=fusion, scope=my_scope, reranker=reranker)
    reranked_top2 = top_docs(reranked_run, k=2)
    verdict = "찾음" if MY_EXPECTED_DOC in reranked_top2 else "못 찾음"
    print(f"재순위  | 상위 2개 문서={reranked_top2} -> 기대 문서({MY_EXPECTED_DOC}): {verdict}")

마지막 **세 번째 실습**에서는 내 질문(`MY_QUESTION`)을 Decision API 분해 관문(`needs_decomposition`)에 넣어 봅니다. 단일 사실 질문일 때와 두 문서의 사실을 엮은 복합 질문일 때 관문 판정(`P`)이 어떻게 달라지는지 직접 질문을 바꾸어 실험해 보세요.

In [ ]:
# [내 것 만들기] 내 질문이 분해 관문을 통과하면 하위 질의로 나누어 합치고 원 질문으로 재순위합니다 (Jev 1회, 열리면 Claude 1회)
MY_QUESTION = "출장 가서 하룻밤 자면 숙박비를 얼마까지 낼 수 있나요?"

if not MY_QUESTION.strip():
    print("MY_QUESTION에 질문을 채워 주세요.")
else:
    # [1단계] 내 질문의 범위(Scope)를 만들고 Decision API 분해 관문에 질문을 넣습니다
    my_scope = Scope(MY_ENTITY, date.fromisoformat(MY_AS_OF))
    print("[내 질문의 관문] Decision API(Jev) 호출 1회, Claude 호출은 관문이 열릴 때만 1회입니다. 진행합니다.")
    my_gate = needs_decomposition(decider, MY_QUESTION)
    my_gate_text = "열림(분해 실행)" if my_gate.decision else "닫힘(분해 생략)"
    print(f"관문 판정: P={my_gate.probability}, 문턱값={my_gate.threshold} -> {my_gate_text}")

    # [2단계] 관문이 열렸을 때만 Claude를 호출해 하위 질의로 분해합니다
    my_queries = []
    if my_gate.decision:
        announce("내 질문을 하위 질의로 분해합니다", model_calls=1)
        my_queries = decompose_question(make_model(), MY_QUESTION)
    print("분해된 하위 질의:", my_queries or "(분해하지 않음)")

    # [3단계] 원 질문만 검색한 결과와 하위 질의를 합쳐 검색한 결과의 상위 2개 문서를 비교합니다
    plain_run = retrieve(MY_QUESTION, retriever=fusion, scope=my_scope, reranker=reranker)
    split_run = retrieve(
        MY_QUESTION,
        retriever=fusion,
        scope=my_scope,
        reranker=reranker,
        queries=my_queries or None,
    )
    print("원 질문만 검색 상위 2개:", top_docs(plain_run, k=2))
    print("분해 합침 검색 상위 2개:", top_docs(split_run, k=2))

## 확인 질문 (3분)

1. 다섯 문장(`N=5`) 가운데 `'식당'`이라는 토큰이 두 문장(`df=2`)에 들어 있을 때, 우리 `BM25Index` 공식이 계산하는 IDF 식과 값을 말해 보세요.
2. 4절 `[바꿔보기]` 셀의 출력 값을 근거로, 이 질의에서 Dense 1위 문장을 답변의 인용 근거로 그대로 써도 되는지 판단해 보세요. 그렇게 판단한 근거가 된 출력 숫자는 무엇인가요?
3. 7절 `[깨뜨리기]` 셀 출력의 두 줄을 비교해, 사후 필터에서 자리를 잃어버린 문서가 무엇이고 어느 단계에서 탈락했는지 설명해 보세요. 또한 상위 `k`개를 더 넓게 뽑더라도 사후 필터가 근본적으로 안전하지 않은 이유는 무엇인가요?
4. 10절 `POOLS` `[바꿔보기]` 출력에서 후보가 1개, 2개일 때의 `범위 안` `r@3` 값을 읽고, 왜 교차 인코더 재순위가 그 값을 절대 넘을 수 없는지 설명해 보세요. 또한 우리 수업 문서에서 후보 18개와 50개의 결과가 같게 나왔더라도, 어떤 조건의 말뭉치에서는 후보 50개가 꼭 필요한가요?
5. 12절에서 인용한 Ko-StrategyQA의 `0.835`와 올거나이즈 `tune`의 `0.714 대 0.762`를 같은 줄에 놓고 단순 비교하면 안 되는 이유는 무엇인가요? 두 벤치마크 사이에서 1위 구성이 뒤집힌 원인 후보 가운데 이 노트북에서 실제로 측정해 확인한 것은 무엇인가요?
6. 11절 관문 셀 출력의 `P`(실행할 때마다 조금씩 달라집니다)를 `'이 질문을 분해해야 할 실제 확률'`로 곧바로 읽으면 어떤 점이 틀리나요? 그리고 이 관문을 우리 회사의 실제 내부 질문에 적용하기 전에 보안·운영 면에서 반드시 먼저 확인해야 할 점은 무엇인가요?
7. 11절 `[바꿔보기]` 출력의 `합쳐진 후보` 수와 `현재 Scope가 허용한 조각` 수를 바탕으로, 재순위 후보 한도(`pool=50`)가 어떤 조건(부등식)일 때만 질문 분해가 최종 재순위 결과를 바꿀 수 있는지 적어 보세요. 우리 수업 문서에서 질문 분해와 출처 보강을 기본으로 켜 둔 근거는 '측정된 성능 향상'인가요, 아니면 '아키텍처 설계 선택'인가요?

## 정리 (2분)

- **근거 단위와 범위 필터**: 표가 중간에 잘리지 않도록 절 단위로 청킹하는 규칙을 지키고, 고객(`entity`)·기준일(`as_of`)·승인 상태 같은 메타데이터는 관련도가 아니라 자격이므로 검색 점수를 매기기 **전에** 먼저 거릅니다.
- **넓은 후보와 원 질문 재순위**: 1차 검색 후보는 점수 융합(`fusion`)으로 넓게 모으고, 교차 인코더 재순위는 항상 **원 질문** 기준으로 수행하며, 재순위 모델이 실패하더라도 1차 검색 순서를 유지하고 경고를 남깁니다. 초기 후보 깊이(`pool`)가 재현율(recall)의 천장입니다.
- **Jev 관문과 출처 보강**: 질문 분해는 Decision API 관문 뒤에서, 출처 보강은 `Retriever`에서 기본으로 켜 두었습니다. 관문의 점수 `P`는 보정된 확률이 아니며 문턱값은 코드 규칙입니다. 질문 분해의 이득은 질문과 정답 문서의 구조에 달려 있고, 출처 보강은 재순위가 정상일 때는 차이가 없었으며 재순위가 실패할 때는 오히려 성능이 낮아졌습니다. 두 기능을 켜 둔 이유는 측정 우위가 아니라 AutoRAG를 따르는 설계 선택입니다.
- **정직한 측정**: 검색 방식의 우열은 일반론이 아니라 **측정**으로 확인하며, 실패하거나 누락된 행을 결과에서 숨기거나 지우지 않습니다.
- **PDF 쪽 판독**: 글자층이 없는 차트나 스캔 쪽은 멀티모달 모델로 판독해 색인해야 후보로 찾을 수 있지만, 판독된 수치를 인용할 때는 원본 쪽과의 대조가 따로 필요합니다. 다음 노트북(`05-Production-Agentic-RAG.ipynb`)에서는 이 검색 파이프라인을 운영형 Agent의 안전장치들과 결합합니다.